# GraphFix: Adaptive Graph-Guided Context Selection for Local Software Engineering Agents

## Research Question

Can adaptive code-graph-guided context selection improve the localization of repository-relevant code while reducing the amount of irrelevant context supplied to a local software engineering agent?

## Hypothesis

Semantic retrieval can identify code that is lexically or semantically related to a software issue, but relevant software behavior is often distributed across callers, callees, dependencies, and related symbols.

We hypothesize that combining semantic retrieval with targeted graph expansion can improve relevant-code coverage without requiring the agent to process the entire repository.

## Proposed Approach

GraphFix first retrieves candidate code nodes using the supplied code embeddings. It then expands the candidate set through the repository's code graph.

Rather than including every neighboring node, GraphFix applies an adaptive selection procedure that ranks candidate nodes according to semantic relevance and graph proximity while limiting redundant context.

The selected context is intended to provide a local software engineering agent with the smallest useful subgraph for solving the issue.

## Experimental Comparison

We will compare four context-selection strategies:

1. Semantic retrieval only
2. Graph expansion only
3. Semantic retrieval with fixed graph expansion
4. GraphFix with adaptive graph-guided selection

## Current Status

The dataset, repository graphs, code embeddings, and task metadata are being inspected before implementing the retrieval and graph-selection experiments.

No experimental results are claimed at this stage.

In [10]:
import os

BASE = "/kaggle/input"

for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, "").count(os.sep)

    if level > 3:
        dirs[:] = []

    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        print(f"{indent}  {file}")

input/
  competitions/
    gemma-4-developer-agent/
      HARNESS_README.md
      tasks.jsonl
      sample_submission/
        eval_config.yaml
        agent.yaml
        configs/
          sampling.yaml
        prompts/
          system.md
          analyzer.md
        sub_agents/
          code_analyzer.yaml
        adapters/
      docker/
        Dockerfile.sandbox
        imp.py
        telnetlib.py
        Dockerfile.public
      snapshots/
        rich_3506.tgz
        rich_4075.tgz
        fastapi_14791.tgz
        fastapi_14463.tgz
        requests_7315.tgz
        fastapi_13920.tgz
        rich_3052.tgz
        requests_6592.tgz
        rich_2943.tgz
        fastapi_15588.tgz
        rich_3296.tgz
        rich_3905.tgz
        fastapi_13713.tgz
        fastapi_14786.tgz
        fastapi_14360.tgz
        fastapi_14873.tgz
        fastapi_14609.tgz
        fastapi_14186.tgz
        rich_3105.tgz
        fastapi_14512.tgz
      wheels/
        packaging-26.3-py3-none-any.whl
    

In [3]:
import json
from pathlib import Path

DATA_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")

tasks_file = DATA_DIR / "tasks.jsonl"

print("Tasks file:", tasks_file)
print("Exists:", tasks_file.exists())

with open(tasks_file, "r", encoding="utf-8") as f:
    first_task = json.loads(f.readline())

print("\nTask fields:")
print(first_task.keys())

print("\nFirst task:")
print(json.dumps(first_task, indent=2)[:8000])

Tasks file: /kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Exists: True

Task fields:
dict_keys(['instance_id', 'repo', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at'])

First task:
{
  "instance_id": "fastapi_15661",
  "repo": "fastapi/fastapi",
  "base_commit": "ee22a4b8ca46dcce26c8c183afc4992a888d8be2",
  "patch": "--- a/scripts/prepare_release.py\n+++ b/scripts/prepare_release.py\n@@ -0,0 +1,216 @@\n+\"\"\"Prepare a release by updating the package version and release notes.\"\"\"\n+\n+import re\n+from datetime import date\n+from pathlib import Path\n+from typing import Annotated, Literal\n+\n+import typer\n+\n+VERSION_PATTERN = re.compile(r'(?m)^__version__ = \"(\\d+\\.\\d+\\.\\d+)\"$')\n+VERSION_HEADING_PATTERN = re.compile(r\"(?m)^## (\\d+\\.\\d+\\.\\d+)(?: \\([^)]+\\))?$\")

In [4]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path("/kaggle/input/competitions/gemma-4-developer-agent")

TASKS_FILE = BASE / "tasks.jsonl"
GRAPHS_DIR = BASE / "graphs"
EMBEDDINGS_DIR = BASE / "embeddings"
SNAPSHOTS_DIR = BASE / "snapshots"

print("Base directory:", BASE)
print("Tasks:", TASKS_FILE.exists())
print("Graphs:", GRAPHS_DIR.exists())
print("Embeddings:", EMBEDDINGS_DIR.exists())
print("Snapshots:", SNAPSHOTS_DIR.exists())

Base directory: /kaggle/input/competitions/gemma-4-developer-agent
Tasks: True
Graphs: True
Embeddings: True
Snapshots: True


In [5]:
tasks = []

with open(TASKS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            tasks.append(json.loads(line))

print("Number of tasks:", len(tasks))

task_df = pd.DataFrame(tasks)

print("\nColumns:")
print(task_df.columns.tolist())

print("\nRepositories:")
print(task_df["repo"].value_counts())

print("\nFirst task:")
print(task_df.iloc[0][
    ["instance_id", "repo", "base_commit", "problem_statement"]
].to_dict())

Number of tasks: 129

Columns:
['instance_id', 'repo', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at']

Repositories:
repo
fastapi/fastapi    67
Textualize/rich    48
psf/requests       13
encode/httpx        1
Name: count, dtype: int64

First task:
{'instance_id': 'fastapi_15661', 'repo': 'fastapi/fastapi', 'base_commit': 'ee22a4b8ca46dcce26c8c183afc4992a888d8be2', 'problem_statement': '👷 Automate release preparation\n\n## Pull Request\r\n\r\n👷 Automate release preparation\r\n\r\n<!--\r\nPlease start with a GitHub Discussion.\r\n\r\nOnce a team member asks you to open a PR, create it and link the discussion here.\r\n\r\nObvious typo fixes can be made in a PR without starting a discussion.\r\n-->\r\n\r\nDiscussion: <!-- Link to the GitHub Discussion -->\r\n\r\n## Description\r\n\r\n<!-- Write the description of your PR here -->\r\n\r\n## AI Disclaimer\r\n\r\nCodex with GPT-5.5, based on https://github.com/fastapi/asyncer/pull/608, reviewed by hand

In [6]:
print("Tasks per repository:")
display(
    task_df.groupby("repo")
    .size()
    .sort_values(ascending=False)
    .to_frame("num_tasks")
)

print("\nProblem statement lengths:")
task_df["problem_length"] = task_df["problem_statement"].fillna("").str.len()

display(
    task_df["problem_length"].describe()
)

Tasks per repository:


,num_tasks
repo,
fastapi/fastapi,67
Textualize/rich,48
psf/requests,13
encode/httpx,1



Problem statement lengths:


count      129.000000
mean       803.573643
std       1195.323098
min         42.000000
25%        146.000000
50%        418.000000
75%       1000.000000
max      10095.000000
Name: problem_length, dtype: float64

In [7]:
graph_files = list(GRAPHS_DIR.glob("*.json"))

print("Number of graph files:", len(graph_files))

for path in graph_files[:3]:
    print("\n" + "=" * 80)
    print("FILE:", path.name)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Python type:", type(data))

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys())[:30])
    elif isinstance(data, list):
        print("Number of items:", len(data))
        if data:
            print("First item:", data[0])

Number of graph files: 127

FILE: rich_36fe3f7ca9becca4777861d5e6e625f5a4a37545.json
Python type: <class 'dict'>
Top-level keys: ['directed', 'multigraph', 'graph', 'nodes', 'edges']

FILE: rich_01b85ac116c4635b6ddb8b488be8254be6df2a84.json
Python type: <class 'dict'>
Top-level keys: ['directed', 'multigraph', 'graph', 'nodes', 'edges']

FILE: fastapi_3c54a8f07b70cdd40e3d81ea319e9fcccd2481d2.json
Python type: <class 'dict'>
Top-level keys: ['directed', 'multigraph', 'graph', 'nodes', 'edges']


In [9]:
# Inspect the structure of one code graph

graph_path = graph_files[0]

with open(graph_path, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

print("Graph:", graph_path.name)
print("Directed:", graph_data["directed"])
print("Multigraph:", graph_data["multigraph"])

print("\nNumber of nodes:", len(graph_data["nodes"]))
print("Number of edges:", len(graph_data["edges"]))

print("\n--- FIRST NODE ---")
print(json.dumps(graph_data["nodes"][0], indent=2)[:5000])

print("\n--- FIRST EDGE ---")
print(json.dumps(graph_data["edges"][0], indent=2)[:5000])

Graph: rich_36fe3f7ca9becca4777861d5e6e625f5a4a37545.json
Directed: True
Multigraph: True

Number of nodes: 2011
Number of edges: 5754

--- FIRST NODE ---
{
  "name": "examples.suppress.hello",
  "text": "def hello(count):\n    \"\"\"Simple program that greets NAME for a total of COUNT times.\"\"\"\n    1 / 0\n    for x in range(count):\n        click.echo(f\"Hello {name}!\")",
  "id": "examples.suppress.hello"
}

--- FIRST EDGE ---
{
  "type": "calls",
  "source": "examples.top_lite_simulator.Process",
  "target": "examples.top_lite_simulator.Process.memory_str",
  "key": 0
}


In [10]:
from collections import Counter

# Count edge relationship types
edge_types = Counter(
    edge.get("type", "UNKNOWN")
    for edge in graph_data["edges"]
)

print("Edge relationship types:")
for edge_type, count in edge_types.most_common():
    print(f"{edge_type:30s} {count:6d}")

Edge relationship types:
calls                            5754


In [11]:
print("Sample graph nodes:\n")

for node in graph_data["nodes"][:10]:
    print("=" * 70)
    print("NAME:", node.get("name"))
    print("TEXT:")
    print(node.get("text", "")[:500])

Sample graph nodes:

NAME: examples.suppress.hello
TEXT:
def hello(count):
    """Simple program that greets NAME for a total of COUNT times."""
    1 / 0
    for x in range(count):
        click.echo(f"Hello {name}!")
NAME: examples.top_lite_simulator.Process
TEXT:
@dataclass
class Process:
    None

    @property
    def memory_str(self) -> str:
        ...

    @property
    def time_str(self) -> str:
        ...
NAME: examples.top_lite_simulator.Process.memory_str
TEXT:
def memory_str(self) -> str:
        if self.memory > 1e6:
            return f"{int(self.memory/1e6)}M"
        if self.memory > 1e3:
            return f"{int(self.memory/1e3)}K"
        return str(self.memory)
NAME: examples.top_lite_simulator.Process.time_str
TEXT:
def time_str(self) -> str:
        return str(datetime.datetime.now() - self.start_time)
NAME: examples.top_lite_simulator.generate_process
TEXT:
def generate_process(pid: int) -> Process:
    return Process(
        pid=pid,
        command=f"Process

In [12]:
print("Graph metadata:")
print(json.dumps(graph_data.get("graph", {}), indent=2)[:5000])

Graph metadata:
{
  "repo_name": "rich_36fe3f7ca9becca4777861d5e6e625f5a4a37545"
}


In [12]:
import os
import json
import glob
import numpy as np
import pandas as pd

BASE = "/kaggle/input/competitions/gemma-4-developer-agent"

# ---------------------------------------------------------
# 1. TASK DATASET
# ---------------------------------------------------------

tasks_path = os.path.join(BASE, "tasks.jsonl")

tasks = []

with open(tasks_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            tasks.append(json.loads(line))

tasks_df = pd.DataFrame(tasks)

print("=" * 70)
print("TASK DATA")
print("=" * 70)

print("Number of tasks:", len(tasks_df))
print("Columns:", list(tasks_df.columns))

print("\nRepositories:")
print(tasks_df["repo"].value_counts())

print("\nExample problem:")
print(tasks_df.iloc[0]["problem_statement"][:1500])

TASK DATA
Number of tasks: 129
Columns: ['instance_id', 'repo', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at']

Repositories:
repo
fastapi/fastapi    67
Textualize/rich    48
psf/requests       13
encode/httpx        1
Name: count, dtype: int64

Example problem:
👷 Automate release preparation

## Pull Request

👷 Automate release preparation

<!--
Please start with a GitHub Discussion.

Once a team member asks you to open a PR, create it and link the discussion here.

Obvious typo fixes can be made in a PR without starting a discussion.
-->

Discussion: <!-- Link to the GitHub Discussion -->

## Description

<!-- Write the description of your PR here -->

## AI Disclaimer

Codex with GPT-5.5, based on https://github.com/fastapi/asyncer/pull/608, reviewed by hand.

<!-- If using AI, write here the prompt and model used -->

<details>
<summary>AI transcript</summary>

<!-- Paste here the entire AI transcript -->

</details>

## Checklist

- [ ] This

In [14]:
# ---------------------------------------------------------
# 2. GRAPH DATASET
# ---------------------------------------------------------

GRAPH_DIR = os.path.join(BASE, "graphs")
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.json")))

print("=" * 70)
print("GRAPH DATA")
print("=" * 70)

print("Number of graph files:", len(graph_files))

# Inspect first graph
with open(graph_files[0], "r", encoding="utf-8") as f:
    graph = json.load(f)

print("\nGraph:", os.path.basename(graph_files[0]))

print("Top-level keys:", graph.keys())
print("Nodes:", len(graph["nodes"]))
print("Edges:", len(graph["edges"]))

print("\nFirst node:")
print(json.dumps(graph["nodes"][0], indent=2)[:2000])

print("\nFirst 5 edges:")
for edge in graph["edges"][:5]:
    print(edge)

GRAPH DATA
Number of graph files: 127

Graph: fastapi_016ab760faba1813ba235d5582a014a58bd2f417.json
Top-level keys: dict_keys(['directed', 'multigraph', 'graph', 'nodes', 'edges'])
Nodes: 4287
Edges: 2490

First node:
{
  "name": "path_operation_advanced_configuration.tutorial004_py310.Item",
  "text": "class Item(BaseModel):\n    None",
  "id": "path_operation_advanced_configuration.tutorial004_py310.Item"
}

First 5 edges:
{'type': 'calls', 'source': 'settings.app02_py310.main.get_settings', 'target': 'settings.app02_py310.config.Settings', 'key': 0}
{'type': 'calls', 'source': 'settings.app02_py310.config.Settings', 'target': 'settings.app02_py310.main.get_settings', 'key': 0}
{'type': 'calls', 'source': 'settings.app02_py310.config.Settings', 'target': 'settings.app02_py310.test_main.get_settings_override', 'key': 0}
{'type': 'calls', 'source': 'settings.app02_py310.test_main.get_settings_override', 'target': 'settings.app02_py310.config.Settings', 'key': 0}
{'type': 'calls', 'sour

In [15]:
# ---------------------------------------------------------
# 3. EDGE TYPE ANALYSIS
# ---------------------------------------------------------

from collections import Counter

edge_types = Counter()

for path in graph_files:
    with open(path, "r", encoding="utf-8") as f:
        g = json.load(f)

    for edge in g["edges"]:
        edge_types[edge.get("type", "UNKNOWN")] += 1

print("=" * 70)
print("EDGE TYPES")
print("=" * 70)

for edge_type, count in edge_types.most_common():
    print(f"{edge_type:30s} {count:,}")

EDGE TYPES
calls                          452,588


In [16]:
# ---------------------------------------------------------
# 4. EMBEDDING STRUCTURE
# ---------------------------------------------------------

EMBED_DIR = os.path.join(BASE, "embeddings")
embedding_files = sorted(glob.glob(os.path.join(EMBED_DIR, "*.npz")))

print("=" * 70)
print("EMBEDDINGS")
print("=" * 70)

print("Number of embedding files:", len(embedding_files))

sample_embedding = np.load(embedding_files[0], allow_pickle=True)

print("\nFile:", os.path.basename(embedding_files[0]))
print("Keys:", sample_embedding.files)

for key in sample_embedding.files:
    arr = sample_embedding[key]
    print(
        f"{key}: shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )

EMBEDDINGS
Number of embedding files: 127

File: fastapi_016ab760faba1813ba235d5582a014a58bd2f417.npz
Keys: ['path_operation_advanced_configuration.tutorial004_py310.Item', 'path_operation_advanced_configuration.tutorial002_py310.use_route_names_as_operation_ids', 'path_operation_advanced_configuration.tutorial006_py310.magic_data_reader', 'path_operation_advanced_configuration.tutorial007_py310.Item', 'settings.tutorial001_py310.Settings', 'settings.app02_py310.main.get_settings', 'settings.app02_py310.config.Settings', 'settings.app02_py310.test_main.get_settings_override', 'settings.app02_py310.test_main.test_app', 'settings.app02_an_py310.main.get_settings', 'settings.app02_an_py310.config.Settings', 'settings.app02_an_py310.test_main.get_settings_override', 'settings.app02_an_py310.test_main.test_app', 'settings.app01_py310.config.Settings', 'settings.app03_an_py310.config.Settings', 'settings.app03_an_py310.main.get_settings', 'settings.app03_py310.main.get_settings', 'settings.a

In [17]:
# ---------------------------------------------------------
# 5. GRAPH REPOSITORY DISTRIBUTION
# ---------------------------------------------------------

graph_repo_counts = Counter()

graph_stats = []

for path in graph_files:

    filename = os.path.basename(path)

    repo_prefix = filename.split("_")[0]

    with open(path, "r", encoding="utf-8") as f:
        g = json.load(f)

    graph_stats.append({
        "file": filename,
        "repo": repo_prefix,
        "nodes": len(g["nodes"]),
        "edges": len(g["edges"]),
    })

    graph_repo_counts[repo_prefix] += 1

graph_stats_df = pd.DataFrame(graph_stats)

print(graph_stats_df.groupby("repo")[["nodes", "edges"]].agg(
    ["count", "mean", "min", "max"]
))

         nodes                          edges                         
         count         mean   min   max count         mean   min   max
repo                                                                  
fastapi     67  3967.537313  3238  4597    67  2457.940299  1988  2780
httpx        1   740.000000   740   740     1  1618.000000  1618  1618
requests    12   735.166667   711   759    12  1824.666667  1748  1884
rich        47  1970.170213  1920  2021    47  5625.361702  5464  5786


In [18]:
# ---------------------------------------------------------
# 6. PATCH LOCALIZATION
# ---------------------------------------------------------

import re

def extract_changed_files(patch):
    if not isinstance(patch, str):
        return []

    files = []

    for match in re.finditer(r"^\+\+\+ b/(.+)$", patch, re.MULTILINE):
        files.append(match.group(1))

    return files


tasks_df["changed_files"] = tasks_df["patch"].apply(extract_changed_files)

print("=" * 70)
print("PATCH LOCALIZATION")
print("=" * 70)

for _, row in tasks_df.head(10).iterrows():
    print("\n", row["instance_id"])
    print("Repo:", row["repo"])
    print("Files:", row["changed_files"])

PATCH LOCALIZATION

 fastapi_15661
Repo: fastapi/fastapi
Files: ['scripts/prepare_release.py']

 fastapi_15588
Repo: fastapi/fastapi
Files: ['fastapi/sse.py']

 fastapi_15589
Repo: fastapi/fastapi
Files: ['fastapi/dependencies/utils.py']

 fastapi_15030
Repo: fastapi/fastapi
Files: ['docs_src/server_sent_events/tutorial001_py310.py', 'docs_src/server_sent_events/tutorial002_py310.py', 'docs_src/server_sent_events/tutorial003_py310.py', 'docs_src/server_sent_events/tutorial004_py310.py', 'docs_src/server_sent_events/tutorial005_py310.py', 'fastapi/openapi/utils.py', 'fastapi/responses.py', 'fastapi/routing.py', 'fastapi/sse.py']

 fastapi_14962
Repo: fastapi/fastapi
Files: ['docs_src/custom_response/tutorial010_py310.py', 'fastapi/_compat/v2.py', 'fastapi/routing.py']

 fastapi_15280
Repo: fastapi/fastapi
Files: ['docs_src/vibe/tutorial001_py310.py', 'fastapi/applications.py']

 fastapi_14964
Repo: fastapi/fastapi
Files: ['fastapi/responses.py']

 fastapi_15023
Repo: fastapi/fastapi
Fil

In [20]:
import os
import glob
import numpy as np

EMBED_DIR = os.path.join(BASE, "competitions", "gemma-4-developer-agent", "embeddings")

embedding_files = sorted(glob.glob(os.path.join(EMBED_DIR, "*.npz")))

print("Number of embedding files:", len(embedding_files))

sample_embedding = np.load(embedding_files[0], allow_pickle=True)

print("\nExample file:")
print(os.path.basename(embedding_files[0]))

print("\nKeys:")
print(sample_embedding.files)

print("\nArrays:")
for key in sample_embedding.files:
    arr = sample_embedding[key]
    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

Number of embedding files: 127

Example file:
fastapi_016ab760faba1813ba235d5582a014a58bd2f417.npz

Keys:
['path_operation_advanced_configuration.tutorial004_py310.Item', 'path_operation_advanced_configuration.tutorial002_py310.use_route_names_as_operation_ids', 'path_operation_advanced_configuration.tutorial006_py310.magic_data_reader', 'path_operation_advanced_configuration.tutorial007_py310.Item', 'settings.tutorial001_py310.Settings', 'settings.app02_py310.main.get_settings', 'settings.app02_py310.config.Settings', 'settings.app02_py310.test_main.get_settings_override', 'settings.app02_py310.test_main.test_app', 'settings.app02_an_py310.main.get_settings', 'settings.app02_an_py310.config.Settings', 'settings.app02_an_py310.test_main.get_settings_override', 'settings.app02_an_py310.test_main.test_app', 'settings.app01_py310.config.Settings', 'settings.app03_an_py310.config.Settings', 'settings.app03_an_py310.main.get_settings', 'settings.app03_py310.main.get_settings', 'settings.app

In [20]:
import os
import re

GRAPH_DIR = os.path.join(BASE, "competitions", "gemma-4-developer-agent", "graphs")

graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.json")))

print("Graph files:", len(graph_files))
print("Tasks:", len(tasks_df))

# Extract repository + commit-like identifier from graph filenames
graph_info = []

for path in graph_files:
    filename = os.path.basename(path)
    stem = filename.replace(".json", "")
    
    parts = stem.split("_", 1)
    
    graph_info.append({
        "filename": filename,
        "repo": parts[0],
        "identifier": parts[1] if len(parts) > 1 else ""
    })

graph_info_df = pd.DataFrame(graph_info)

print("\nGraph distribution:")
print(graph_info_df["repo"].value_counts())

print("\nExample graph identifiers:")
display(graph_info_df.head())

Graph files: 0
Tasks: 129

Graph distribution:


KeyError: 'repo'

In [24]:
print("Example task base commit:")
print(tasks_df.iloc[0]["base_commit"])

print("\nExample graph identifier:")
print(graph_info_df.iloc[0]["identifier"])

Example task base commit:
ee22a4b8ca46dcce26c8c183afc4992a888d8be2

Example graph identifier:
016ab760faba1813ba235d5582a014a58bd2f417


## Mapping Tasks to Repository Snapshots

The task metadata identifies each software-engineering instance by repository and base commit, while the supplied graph and embedding files use repository-specific snapshot identifiers.

Because these identifiers are not directly identical, we first establish the correct mapping between tasks, repository snapshots, graphs, and embeddings before running retrieval experiments.

This mapping step is important for ensuring that each task is evaluated against the correct version of its repository and that localization metrics do not use information from an incorrect snapshot.

In [25]:
# Inspect graph metadata to understand how snapshots are identified

sample_graph_path = graph_files[0]

with open(sample_graph_path, "r", encoding="utf-8") as f:
    sample_graph = json.load(f)

print("Graph file:")
print(os.path.basename(sample_graph_path))

print("\nGraph metadata:")
print(json.dumps(sample_graph.get("graph", {}), indent=2))

print("\nFirst node:")
print(json.dumps(sample_graph["nodes"][0], indent=2)[:1500])

print("\nNumber of nodes:", len(sample_graph["nodes"]))
print("Number of edges:", len(sample_graph["edges"]))

Graph file:
fastapi_016ab760faba1813ba235d5582a014a58bd2f417.json

Graph metadata:
{
  "repo_name": "fastapi_016ab760faba1813ba235d5582a014a58bd2f417"
}

First node:
{
  "name": "path_operation_advanced_configuration.tutorial004_py310.Item",
  "text": "class Item(BaseModel):\n    None",
  "id": "path_operation_advanced_configuration.tutorial004_py310.Item"
}

Number of nodes: 4287
Number of edges: 2490


## Repository Snapshot Representation

The supplied repository graphs are identified using repository-specific snapshot identifiers rather than the task metadata's base commit SHA.

For example, a FastAPI graph is represented as:

`fastapi_016ab760faba1813ba235d5582a014a58bd2f417`

while the corresponding task metadata contains a separate Git base commit SHA.

Therefore, GraphFix will use the competition's provided repository snapshot resources and their associated graph/embedding artifacts rather than assuming that graph identifiers are Git commit hashes.

In [26]:
id="k8m2ra"
from pathlib import Path

readme_path = DATA_DIR / "HARNESS_README.md"

print("README exists:", readme_path.exists())

if readme_path.exists():
    readme_text = readme_path.read_text(encoding="utf-8")
    
    # Show sections mentioning graphs, embeddings, snapshots, or repository IDs
    lines = readme_text.splitlines()
    
    keywords = ["graph", "embedding", "snapshot", "repo", "commit"]
    
    for i, line in enumerate(lines):
        if any(k in line.lower() for k in keywords):
            start = max(0, i - 2)
            end = min(len(lines), i + 5)
            print("\n" + "=" * 80)
            print("\n".join(lines[start:end]))

README exists: True

```mermaid
flowchart TB
    subgraph Submission["Competitor Submission (submission.zip / agent_dir)"]
        YAML["agent.yaml + sub_agents/*.yaml"]
        EvalCfg["eval_config.yaml (Optional Budgets)"]
        Prompts["prompts/*.md, configs/*.yaml"]
        Adapters["adapters/* (Optional LoRA / Weights)"]

    end

    subgraph Host["Harness Process (swegemma + adk-submission + adk-eval-core)"]
        Validator["validate_directory & validate_single_declared_model"]
        Server["Local Inference Server (vLLM / Transformers :8000)"]
        Compiler["compile_submission() -> ADK Runner"]
        Tools["SwegemmaContext (9 Bound Tools + Budget Gate)"]

    end

    subgraph Sandboxes["Isolated Per-Task Sandboxes"]
        ContA["Container A: Agent Sandbox (/workspace)\nSnapshot + Editable Install + Baseline Commit"]
        ContB["Container B: Verification Sandbox (/workspace)\nFresh Snapshot + agent_patch + test_patch + pytest"]
    end



    subgraph Sandboxes["

## Code Intelligence Resources

The competition provides pre-computed code intelligence resources for each supported repository.

These resources include:

- Code embeddings for semantic similarity search
- A repository call/dependency graph
- Graph-neighbor traversal
- Induced subgraph extraction

GraphFix uses these resources as the foundation for adaptive context selection. Instead of treating semantic retrieval and graph traversal as separate navigation strategies, GraphFix will combine them to construct a focused set of repository code relevant to each software-engineering task.

The experiments will compare semantic retrieval, graph-based expansion, and adaptive graph-guided selection using the same underlying repository resources.

In [27]:
# Inspect the embedding archive structure

EMBED_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "embeddings"
)

embedding_files = sorted(glob.glob(os.path.join(EMBED_DIR, "*.npz")))

print("Number of embedding files:", len(embedding_files))

sample_embedding_path = embedding_files[0]

print("\nExample embedding file:")
print(os.path.basename(sample_embedding_path))

sample_embedding = np.load(
    sample_embedding_path,
    allow_pickle=True
)

print("\nKeys:")
print(sample_embedding.files)

print("\nContents:")
for key in sample_embedding.files:
    arr = sample_embedding[key]
    print(
        f"{key}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )

Number of embedding files: 127

Example embedding file:
fastapi_016ab760faba1813ba235d5582a014a58bd2f417.npz

Keys:
['path_operation_advanced_configuration.tutorial004_py310.Item', 'path_operation_advanced_configuration.tutorial002_py310.use_route_names_as_operation_ids', 'path_operation_advanced_configuration.tutorial006_py310.magic_data_reader', 'path_operation_advanced_configuration.tutorial007_py310.Item', 'settings.tutorial001_py310.Settings', 'settings.app02_py310.main.get_settings', 'settings.app02_py310.config.Settings', 'settings.app02_py310.test_main.get_settings_override', 'settings.app02_py310.test_main.test_app', 'settings.app02_an_py310.main.get_settings', 'settings.app02_an_py310.config.Settings', 'settings.app02_an_py310.test_main.get_settings_override', 'settings.app02_an_py310.test_main.test_app', 'settings.app01_py310.config.Settings', 'settings.app03_an_py310.config.Settings', 'settings.app03_an_py310.main.get_settings', 'settings.app03_py310.main.get_settings', 'se

In [28]:
# Check whether embedding keys match graph node IDs

import json
import os
import numpy as np

GRAPH_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "graphs"
)

# Load the graph corresponding to the embedding file
sample_graph_path = os.path.join(
    GRAPH_DIR,
    "fastapi_016ab760faba1813ba235d5582a014a58bd2f417.json"
)

with open(sample_graph_path, "r") as f:
    graph_data = json.load(f)

graph_nodes = set(
    node["id"]
    for node in graph_data["nodes"]
)

embedding_keys = set(
    sample_embedding.files
)

print("Graph nodes:", len(graph_nodes))
print("Embedding keys:", len(embedding_keys))

common = graph_nodes & embedding_keys

print("Matching IDs:", len(common))

print(
    "Graph → Embedding match:",
    len(common) / len(graph_nodes)
)

print(
    "Embedding → Graph match:",
    len(common) / len(embedding_keys)
)

print("\nExample matching nodes:")
for node in list(common)[:10]:
    print(node)

print("\nEmbedding-only examples:")
for node in list(embedding_keys - graph_nodes)[:10]:
    print(node)

print("\nGraph-only examples:")
for node in list(graph_nodes - embedding_keys)[:10]:
    print(node)

Graph nodes: 4287
Embedding keys: 4287
Matching IDs: 4287
Graph → Embedding match: 1.0
Embedding → Graph match: 1.0

Example matching nodes:
tests.test_request_params.test_form.test_list.FormModelRequiredListStr
tests.test_tutorial.test_separate_openapi_schemas.test_tutorial001.test_read_items
tests.test_tutorial.test_query_params_str_validations.test_tutorial002.test_query_params_str_validations_q_too_long
tests.test_response_model_as_return_annotation.test_response_model_no_annotation_return_exact_dict
tests.test_tutorial.test_header_params.test_tutorial003.test
tests.test_extra_routes.get_items
tests.benchmarks.test_general_performance.test_sync_return_large_dict_with_response_model
tests.test_tutorial.test_security.test_tutorial005.test_token_no_sub
tests.test_tutorial.test_python_types.test_tutorial008.test_process_items
tests.test_response_by_alias.no_alias_model

Embedding-only examples:

Graph-only examples:


In [35]:
# Inspect snapshot files and resource identifiers

import os
import glob

SNAPSHOT_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "snapshots"
)

GRAPH_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "graphs"
)

EMBED_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "embeddings"
)

snapshot_files = sorted(glob.glob(os.path.join(SNAPSHOT_DIR, "*")))
graph_files = sorted(glob.glob(os.path.join(GRAPH_DIR, "*.json")))
embedding_files = sorted(glob.glob(os.path.join(EMBED_DIR, "*.npz")))

print("Snapshots:", len(snapshot_files))
print("Graphs:", len(graph_files))
print("Embeddings:", len(embedding_files))

print("\nSample snapshot files:")
for f in snapshot_files[:10]:
    print(os.path.basename(f))

print("\nSample graph files:")
for f in graph_files[:10]:
    print(os.path.basename(f))

print("\nSample embedding files:")
for f in embedding_files[:10]:
    print(os.path.basename(f))

Snapshots: 129
Graphs: 127
Embeddings: 127

Sample snapshot files:
fastapi_11194.tgz
fastapi_11355.tgz
fastapi_12942.tgz
fastapi_13207.tgz
fastapi_13537.tgz
fastapi_13713.tgz
fastapi_13786.tgz
fastapi_13920.tgz
fastapi_14077.tgz
fastapi_14099.tgz

Sample graph files:
fastapi_016ab760faba1813ba235d5582a014a58bd2f417.json
fastapi_08924400c2c75533685148b029defcf6913f1def.json
fastapi_08b09e5236e315b6f10265ed229f130d4befb4ea.json
fastapi_0bdc3ca3732e3742e094bea2b320e9774659ac38.json
fastapi_0dd42b746e17d33a74ea267a62eace9ccdccf345.json
fastapi_16d75d90eb96976a57b94cc24e4018859cd54c4d.json
fastapi_1b3bea8b6be2220cc1544653ea785d2134fcb35b.json
fastapi_1e78a36b7310003f0ff634627d8a7bc53c6ccdee.json
fastapi_1fcec88ad2df4541b6a81c4786c4d039dc745e1e.json
fastapi_202b2d2f5f331db9102b5dbcef071a9e09bed10e.json

Sample embedding files:
fastapi_016ab760faba1813ba235d5582a014a58bd2f417.npz
fastapi_08924400c2c75533685148b029defcf6913f1def.npz
fastapi_08b09e5236e315b6f10265ed229f130d4befb4ea.npz
fastapi_

In [36]:
# Inspect one snapshot archive

import tarfile
import os
import glob

sample_snapshot = snapshot_files[0]

print("Snapshot:", os.path.basename(sample_snapshot))

with tarfile.open(sample_snapshot, "r:gz") as tar:
    members = tar.getmembers()

print("Number of files:", len(members))

print("\nFirst 30 files:")
for member in members[:30]:
    print(member.name)

Snapshot: fastapi_11194.tgz
Number of files: 2941

First 30 files:
.
./.git
./.git/HEAD
./.git/config
./.git/description
./.git/hooks
./.git/hooks/applypatch-msg.sample
./.git/hooks/commit-msg.sample
./.git/hooks/fsmonitor-watchman.sample
./.git/hooks/post-update.sample
./.git/hooks/pre-applypatch.sample
./.git/hooks/pre-commit.sample
./.git/hooks/pre-merge-commit.sample
./.git/hooks/pre-push.sample
./.git/hooks/pre-rebase.sample
./.git/hooks/pre-receive.sample
./.git/hooks/prepare-commit-msg.sample
./.git/hooks/push-to-checkout.sample
./.git/hooks/sendemail-validate.sample
./.git/hooks/update.sample
./.git/index
./.git/info
./.git/info/exclude
./.git/logs
./.git/logs/HEAD
./.git/logs/refs
./.git/logs/refs/heads
./.git/logs/refs/heads/export_ref
./.git/logs/refs/heads/main
./.git/objects


In [40]:
# Inspect Git metadata inside the snapshot without extracting everything

import tarfile
import os

sample_snapshot = snapshot_files[0]

with tarfile.open(sample_snapshot, "r:gz") as tar:
    names = tar.getnames()

print("Snapshot:", os.path.basename(sample_snapshot))

print("\nGit HEAD-related files:")
for name in names:
    if name.endswith("/.git/HEAD") or name == ".git/HEAD":
        print(name)

print("\nGit config-related files:")
for name in names:
    if name.endswith("/.git/config") or name == ".git/config":
        print(name)

print("\nPotential Git refs:")
count = 0
for name in names:
    if "/.git/refs/" in name or name.startswith(".git/refs/"):
        print(name)
        count += 1
        if count >= 20:
            break

Snapshot: fastapi_11194.tgz

Git HEAD-related files:
./.git/HEAD

Git config-related files:
./.git/config

Potential Git refs:
./.git/refs/heads
./.git/refs/heads/export_ref
./.git/refs/heads/main
./.git/refs/tags


In [42]:
# Read Git commit information directly from the snapshot archive

import tarfile
import os

sample_snapshot = snapshot_files[0]

with tarfile.open(sample_snapshot, "r:gz") as tar:

    def read_file(path):
        member = tar.getmember(path)
        return tar.extractfile(member).read().decode("utf-8").strip()

    head = read_file("./.git/HEAD")

    print("Snapshot:", os.path.basename(sample_snapshot))
    print("HEAD:", head)

    # Read branch refs
    for ref in [
        "./.git/refs/heads/main",
        "./.git/refs/heads/export_ref"
    ]:
        try:
            value = read_file(ref)
            print(f"{ref}: {value}")
        except KeyError:
            print(f"{ref}: not found")

Snapshot: fastapi_11194.tgz
HEAD: ref: refs/heads/main
./.git/refs/heads/main: abd0ef6b5ef05f56d2b3e128ac16b25cd6a6c6ba
./.git/refs/heads/export_ref: abd0ef6b5ef05f56d2b3e128ac16b25cd6a6c6ba


In [45]:
tasks_df["base_commit"]

0      ee22a4b8ca46dcce26c8c183afc4992a888d8be2
1      cb83b83dcf78eab4ea17d504db5abcda705fbdc4
2      22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b
3      48d58ae3b6ef4ded5a4794618c00345667256d98
4      1e78a36b7310003f0ff634627d8a7bc53c6ccdee
                         ...                   
124    1ad6f73a4d1d3b40ac3f879feccfaad66412bd1d
125    0c8bb032ad3b54a32fdb8a43f5ab457c8cd6dcb1
126    01b85ac116c4635b6ddb8b488be8254be6df2a84
127    368e8ad77c95a6bef8f9d332039091d1417296bc
128    aca9467ec6a482b3bed1ba7d2e3de72271c5f817
Name: base_commit, Length: 129, dtype: object

In [47]:
# Determine exactly which task base commits have graph/embedding resources

import os
import glob

GRAPH_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "graphs"
)

EMBED_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "embeddings"
)

# Extract commit IDs from resource filenames
graph_commits = {
    os.path.basename(f).replace(".json", "").split("_", 1)[1]
    for f in glob.glob(os.path.join(GRAPH_DIR, "*.json"))
}

embedding_commits = {
    os.path.basename(f).replace(".npz", "").split("_", 1)[1]
    for f in glob.glob(os.path.join(EMBED_DIR, "*.npz"))
}

task_commits = set(tasks_df["base_commit"])

print("Task base commits:", len(task_commits))
print("Graph commits:", len(graph_commits))
print("Embedding commits:", len(embedding_commits))

# Exact intersections
task_graph = task_commits & graph_commits
task_embedding = task_commits & embedding_commits

print("\nTask ↔ Graph matches:", len(task_graph))
print("Task ↔ Embedding matches:", len(task_embedding))

print("\nTask commits WITHOUT graph:")
for commit in sorted(task_commits - graph_commits):
    row = tasks_df[tasks_df["base_commit"] == commit].iloc[0]
    print(row["instance_id"], "→", commit)

print("\nTask commits WITHOUT embedding:")
for commit in sorted(task_commits - embedding_commits):
    row = tasks_df[tasks_df["base_commit"] == commit].iloc[0]
    print(row["instance_id"], "→", commit)

Task base commits: 127
Graph commits: 127
Embedding commits: 127

Task ↔ Graph matches: 127
Task ↔ Embedding matches: 127

Task commits WITHOUT graph:

Task commits WITHOUT embedding:


In [50]:
# Inspect graph node structure and source-text coverage

import json
import os
import numpy as np

GRAPH_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "graphs"
)

sample_task = tasks_df.iloc[0]
sample_commit = sample_task["base_commit"]
repo_name = sample_task["repo"].split("/")[-1]

graph_path = os.path.join(
    GRAPH_DIR,
    f"{repo_name}_{sample_commit}.json"
)

print("Graph path:")
print(graph_path)

with open(graph_path, "r") as f:
    graph_data = json.load(f)

nodes = graph_data["nodes"]
edges = graph_data["edges"]

print("\nTotal nodes:", len(nodes))
print("Total edges:", len(edges))

print("\nFirst 5 nodes:")
for node in nodes[:5]:
    print(node)

# Source-text coverage
with_text = sum(
    bool(
        isinstance(node.get("text"), str)
        and node.get("text").strip()
    )
    for node in nodes
)

print("\nNodes with source text:", with_text)

print(
    "Text coverage:",
    round(with_text / len(nodes) * 100, 2),
    "%"
)

# Text length statistics
text_lengths = [
    len(node.get("text", ""))
    for node in nodes
    if isinstance(node.get("text"), str)
]

print("\nText length statistics:")
print("Min:", min(text_lengths))
print("Median:", np.median(text_lengths))
print("Mean:", round(np.mean(text_lengths), 2))
print("Max:", max(text_lengths))

# Edge type distribution
edge_types = {}

for edge in edges:
    edge_type = edge.get("type", "unknown")
    edge_types[edge_type] = edge_types.get(edge_type, 0) + 1

print("\nEdge types:")
print(edge_types)

Graph path:
/kaggle/input/competitions/gemma-4-developer-agent/graphs/fastapi_ee22a4b8ca46dcce26c8c183afc4992a888d8be2.json

Total nodes: 4263
Total edges: 2430

First 5 nodes:
{'name': 'path_operation_configuration.tutorial005_py310.Item', 'text': 'class Item(BaseModel):\n    None', 'id': 'path_operation_configuration.tutorial005_py310.Item'}
{'name': 'path_operation_configuration.tutorial001_py310.Item', 'text': 'class Item(BaseModel):\n    None', 'id': 'path_operation_configuration.tutorial001_py310.Item'}
{'name': 'path_operation_configuration.tutorial002_py310.Item', 'text': 'class Item(BaseModel):\n    None', 'id': 'path_operation_configuration.tutorial002_py310.Item'}
{'name': 'path_operation_configuration.tutorial003_py310.Item', 'text': 'class Item(BaseModel):\n    None', 'id': 'path_operation_configuration.tutorial003_py310.Item'}
{'name': 'path_operation_configuration.tutorial004_py310.Item', 'text': 'class Item(BaseModel):\n    None', 'id': 'path_operation_configuration.tut

## Graph Node Representation

Each repository snapshot is represented as a directed code graph.

Each graph node contains:

- A symbolic identifier (`id`)
- A symbolic name (`name`)
- A source-code representation (`text`)

For the inspected FastAPI snapshot, the graph contains thousands of code nodes and call relationships between them.

The node representation allows GraphFix to combine two complementary signals:

1. **Semantic similarity** from the supplied code embeddings
2. **Structural proximity** from the repository code graph

This enables the proposed method to retrieve semantically relevant code and then expand or filter the candidate set using repository structure.

In [24]:
# Baseline 1: Semantic candidate retrieval

import re
import numpy as np

def extract_query_terms(problem_statement):
    """
    Extract code-like terms from a task description.

    We look for:
    - Python identifiers
    - dotted identifiers
    - filenames
    - function/class-like names
    """

    if not isinstance(problem_statement, str):
        return []

    # Dotted names such as:
    # fastapi.routing.APIRoute
    # test_client.get
    dotted = re.findall(
        r'\b[A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)+\b',
        problem_statement
    )

    # Python-style identifiers
    identifiers = re.findall(
        r'\b[A-Za-z_][A-Za-z0-9_]{2,}\b',
        problem_statement
    )

    # Filenames
    filenames = re.findall(
        r'\b[A-Za-z0-9_/-]+\.py\b',
        problem_statement
    )

    terms = dotted + identifiers + filenames

    # Remove common English words
    stopwords = {
        "the", "and", "for", "with", "that", "this",
        "from", "when", "then", "into", "have", "has",
        "should", "would", "could", "there", "their",
        "which", "where", "what", "using", "used",
        "after", "before", "only", "also", "than"
    }

    cleaned = []

    for term in terms:
        if term.lower() not in stopwords:
            cleaned.append(term)

    # Preserve order while removing duplicates
    return list(dict.fromkeys(cleaned))


def cosine_similarity(a, b):
    """Cosine similarity between two vectors."""
    denom = np.linalg.norm(a) * np.linalg.norm(b)

    if denom == 0:
        return 0.0

    return float(np.dot(a, b) / denom)


def load_resources(repo, commit):
    """Load graph and embeddings for a task."""

    repo_name = repo.split("/")[-1]

    graph_path = os.path.join(
        GRAPH_DIR,
        f"{repo_name}_{commit}.json"
    )

    embed_path = os.path.join(
        EMBED_DIR,
        f"{repo_name}_{commit}.npz"
    )

    with open(graph_path, "r") as f:
        graph = json.load(f)

    embeddings = np.load(embed_path)

    return graph, embeddings


def semantic_retrieve(task_row, top_k=20):
    """
    Retrieve graph nodes using symbol/name overlap.

    The supplied embeddings represent code nodes.
    We therefore use task-derived code terms to identify
    candidate symbols before computing semantic similarity.
    """

    graph, embeddings = load_resources(
        task_row["repo"],
        task_row["base_commit"]
    )

    nodes = graph["nodes"]

    query_terms = extract_query_terms(
        task_row["problem_statement"]
    )

    candidates = []

    for node in nodes:

        node_id = node["id"]
        node_name = node.get("name", "")

        # Match task terms against symbolic node names
        matched_terms = [
            term for term in query_terms
            if term.lower() in node_id.lower()
            or term.lower() in node_name.lower()
        ]

        if matched_terms:
            candidates.append(
                (node_id, matched_terms)
            )

    # Rank candidate nodes by number of matched terms
    candidates.sort(
        key=lambda x: len(x[1]),
        reverse=True
    )

    return candidates[:top_k]


# Test on the first task
sample_task = tasks_df.iloc[0]

results = semantic_retrieve(
    sample_task,
    top_k=20
)

print("Task:", sample_task["instance_id"])
print("\nProblem statement:")
print(sample_task["problem_statement"][:1000])

print("\nExtracted query terms:")
print(
    extract_query_terms(
        sample_task["problem_statement"]
    )[:50]
)

print("\nRetrieved candidates:")
for rank, (node_id, matched_terms) in enumerate(results, 1):
    print(
        f"{rank:2d}. {node_id}"
        f" | matched: {matched_terms}"
    )

NameError: name 'EMBED_DIR' is not defined

In [54]:
# Inspect available competition wheels/packages related to embeddings

WHEEL_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent",
    "wheels"
)

print("Wheel directory:")
print(WHEEL_DIR)

wheel_files = sorted(os.listdir(WHEEL_DIR))

print("\nAvailable wheel/package files:")
for filename in wheel_files:
    print(filename)

Wheel directory:
/kaggle/input/competitions/gemma-4-developer-agent/wheels

Available wheel/package files:
Flask-2.2.5-py3-none-any.whl
annotated_types-0.7.0-py3-none-any.whl
annotated_types-0.8.0-py3-none-any.whl
anyio-4.14.1-py3-none-any.whl
anyio-4.14.2-py3-none-any.whl
certifi-2026.6.17-py3-none-any.whl
certifi-2026.7.22-py3-none-any.whl
charset_normalizer-3.4.7-py3-none-any.whl
charset_normalizer-3.4.9-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
charset_normalizer-3.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
click-8.2.1-py3-none-any.whl
click-8.4.2-py3-none-any.whl
colorama-0.4.6-py2.py3-none-any.whl
editables-0.6-py3-none-any.whl
fastapi-0.141.1-py3-none-any.whl
flask-2.3.3-py3-none-any.whl
flask-3.1.3-py3-none-any.whl
flit_core-3.12.0-py3-none-any.whl
hatchling-1.30.1-py3-none-any.whl
httpcore-1.0.9-py3-none-any.whl
httpx-0.28.1-py3-none-any.whl
idna-3.18-py3-none-any.whl
idna-3.19-py3-none-any.whl
in

In [55]:
# Inspect competition source files for code-intelligence implementation

import os

COMP_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent"
)

print("Competition directory:")
print(COMP_DIR)

for root, dirs, files in os.walk(COMP_DIR):
    # Skip large/unimportant resource directories
    dirs[:] = [d for d in dirs if d not in ["snapshots", "graphs", "embeddings", "wheels", "docker"]]

    for f in files:
        if f.endswith((".py", ".md", ".txt")):
            path = os.path.join(root, f)
            print(path.replace(COMP_DIR, ""))

Competition directory:
/kaggle/input/competitions/gemma-4-developer-agent
/HARNESS_README.md
/sample_submission/prompts/system.md
/sample_submission/prompts/analyzer.md
/sandbox/setup.py


In [56]:
# Inspect code-like identifiers mentioned in task descriptions

import re

def extract_identifiers(text):
    if not isinstance(text, str):
        return []

    # Python/module/function/class-like identifiers
    patterns = [
        r'\b[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_.]*\b',
        r'\b[A-Za-z_][A-Za-z0-9_]*\(\)',
        r'\b[A-Z][A-Za-z0-9_]{2,}\b',
        r'\b[a-z_][a-z0-9_]{2,}\b'
    ]

    found = []

    for pattern in patterns:
        found.extend(re.findall(pattern, text))

    # Remove obvious generic English words
    stopwords = {
        "the", "and", "for", "with", "this", "that",
        "from", "have", "has", "will", "would", "should",
        "please", "when", "where", "what", "which",
        "pull", "request", "issue", "github", "test",
        "tests", "change", "changes", "file", "files",
        "code", "use", "using", "add", "added", "new",
        "fix", "fixed", "update", "updated"
    }

    cleaned = []

    for x in found:
        x = x.strip(".,:;()[]{}<>`'\"")

        base = x.rstrip("()").split(".")[-1].lower()

        if len(base) >= 3 and base not in stopwords:
            cleaned.append(x.rstrip("()"))

    # preserve order, remove duplicates
    return list(dict.fromkeys(cleaned))


for idx in range(min(10, len(tasks_df))):
    row = tasks_df.iloc[idx]

    print("=" * 80)
    print("INSTANCE:", row["instance_id"])
    print("REPO:", row["repo"])
    print("CHANGED FILES:", row["changed_files"])

    identifiers = extract_identifiers(row["problem_statement"])

    print("\nPossible identifiers:")
    print(identifiers[:50])

INSTANCE: fastapi_15661
REPO: fastapi/fastapi
CHANGED FILES: ['scripts/prepare_release.py']

Possible identifiers:
['github.com', 'Automate', 'Discussion', 'Once', 'Obvious', 'Link', 'Description', 'Write', 'Disclaimer', 'Codex', 'GPT', 'Paste', 'Checklist', 'Coverage', 'release', 'preparation', 'start', 'team', 'member', 'asks', 'you', 'open', 'create', 'link', 'discussion', 'here', 'typo', 'fixes', 'can', 'made', 'without', 'starting', 'description', 'your', 'based', 'https', 'com', 'fastapi', 'asyncer', 'reviewed', 'hand', 'write', 'prompt', 'model', 'used', 'details', 'summary', 'transcript', 'entire', 'obvious']
INSTANCE: fastapi_15588
REPO: fastapi/fastapi
CHANGED FILES: ['fastapi/sse.py']

Possible identifiers:
['Validate', 'Server', 'Sent', 'Event', 'Discussion', 'Once', 'Obvious', 'Link', 'Description', 'FastAPI', 'SSE', 'Write', 'Disclaimer', 'Codex', 'GPT', 'Paste', 'Checklist', 'Coverage', 'fields', 'avoid', 'applications', 'sending', 'broken', 'data', 'start', 'team', 'mem

In [57]:
# Build a code-aware symbol matching baseline

import re
import numpy as np
from collections import defaultdict

# Generic words that should not influence code-symbol matching
GENERIC_WORDS = {
    "the", "and", "for", "with", "this", "that", "from",
    "have", "has", "will", "would", "should", "could",
    "please", "when", "where", "what", "which",
    "once", "start", "starting", "write", "description",
    "discussion", "link", "github", "request", "pull",
    "issue", "test", "tests", "change", "changes",
    "file", "files", "code", "use", "using", "used",
    "add", "added", "new", "fix", "fixed", "update",
    "updated", "support", "available", "released",
    "there", "here", "need", "only", "can", "made",
    "without", "your", "you", "was", "were", "not",
    "server", "response", "model", "data", "application",
    "applications", "default", "return", "type", "true",
    "false", "json", "html"
}


def extract_code_candidates(text):
    """
    Extract tokens that look like real programming identifiers.
    """
    if not isinstance(text, str):
        return []

    candidates = set()

    # Qualified identifiers: app.vibe, module.function, etc.
    qualified = re.findall(
        r'\b[A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)+\b',
        text
    )

    # snake_case identifiers
    snake = re.findall(
        r'\b[A-Za-z_][A-Za-z0-9]*_[A-Za-z0-9_]+\b',
        text
    )

    # CamelCase / class-like identifiers
    camel = re.findall(
        r'\b[A-Z][A-Za-z0-9]{2,}\b',
        text
    )

    for item in qualified + snake + camel:
        item = item.strip(".,:;()[]{}<>`'\"")

        if len(item) < 3:
            continue

        if item.lower() in GENERIC_WORDS:
            continue

        candidates.add(item)

    return sorted(candidates)


def score_symbol_match(query, node_id, node_name):
    """
    Score how strongly an extracted issue identifier
    matches a graph symbol.
    """
    q = query.lower()
    nid = node_id.lower()
    nname = node_name.lower()

    score = 0

    # Exact identifier match
    if q == nid:
        score += 100

    if q == nname:
        score += 90

    # Exact component match
    node_parts = re.split(r"[.:/]", nid)

    if q in node_parts:
        score += 70

    # Strong substring match
    if q in nid:
        score += 40

    if q in nname:
        score += 30

    # Prefer longer/more specific identifiers
    score += min(len(q), 30)

    return score


def symbol_retrieve(task_row, top_k=20):
    """
    Retrieve graph nodes using code-like identifiers
    extracted from the task description.
    """

    repo = task_row["repo"]
    commit = task_row["base_commit"]

    graph, embeddings = load_resources(repo, commit)

    candidates = extract_code_candidates(
        task_row["problem_statement"]
    )

    results = []

    for node in graph["nodes"]:
        node_id = str(node.get("id", ""))
        node_name = str(node.get("name", ""))

        best_score = 0
        best_query = None

        for query in candidates:
            score = score_symbol_match(
                query,
                node_id,
                node_name
            )

            if score > best_score:
                best_score = score
                best_query = query

        if best_score > 0:
            results.append({
                "node_id": node_id,
                "node_name": node_name,
                "score": best_score,
                "matched_query": best_query
            })

    results.sort(
        key=lambda x: (-x["score"], x["node_id"])
    )

    return candidates, results[:top_k]


# Test on several tasks
test_instances = [
    "fastapi_15589",
    "fastapi_14964",
    "fastapi_15280",
    "fastapi_14978",
    "fastapi_15661"
]

for instance_id in test_instances:

    row = tasks_df[
        tasks_df["instance_id"] == instance_id
    ].iloc[0]

    candidates, results = symbol_retrieve(row, top_k=10)

    print("\n" + "=" * 90)
    print("INSTANCE:", instance_id)
    print("CHANGED FILES:", row["changed_files"])

    print("\nCode candidates:")
    print(candidates)

    print("\nTop symbol matches:")
    for r in results:
        print(
            f"{r['score']:>4} | "
            f"{r['matched_query']:<25} | "
            f"{r['node_id']}"
        )


INSTANCE: fastapi_15589
CHANGED FILES: ['fastapi/dependencies/utils.py']

Code candidates:
['Checklist', 'Coverage', 'Disclaimer', 'Obvious', 'Paste', 'convert_underscores']

Top symbol matches:
  78 | Coverage                  | tests.test_dependency_paramless.test_call_get_parameterless_without_scopes_for_coverage
  19 | convert_underscores       | additional_responses.tutorial001_py310.Item
  19 | convert_underscores       | additional_responses.tutorial001_py310.Message
  19 | convert_underscores       | additional_responses.tutorial002_py310.Item
  19 | convert_underscores       | additional_responses.tutorial003_py310.Item
  19 | convert_underscores       | additional_responses.tutorial003_py310.Message
  19 | convert_underscores       | additional_responses.tutorial004_py310.Item
  19 | convert_underscores       | app_testing.app_a_py310.test_main.test_read_main
  19 | convert_underscores       | app_testing.app_b_an_py310.main.Item
  19 | convert_underscores       | app_testin

In [59]:
# Inspect how identifiers are represented in graph node metadata

row = tasks_df[
    tasks_df["instance_id"] == "fastapi_15589"
].iloc[0]

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

queries = [
    "convert_underscores",
    "strict_content_type",
    "ORJSONResponse",
    "app.vibe"
]

for query in queries:

    print("\n" + "=" * 90)
    print("QUERY:", query)

    matches = []

    for node in graph["nodes"]:

        node_id = str(node.get("id", ""))
        node_name = str(node.get("name", ""))

        # ONLY inspect metadata, not source text
        if (
            query.lower() in node_id.lower()
            or query.lower() in node_name.lower()
        ):
            matches.append({
                "id": node_id,
                "name": node_name
            })

    print("Metadata matches:", len(matches))

    for item in matches[:20]:
        print(item)


QUERY: convert_underscores
Metadata matches: 0

QUERY: strict_content_type
Metadata matches: 23
{'id': 'tests.test_strict_content_type_nested.test_strict_inner_on_lax_app_rejects_no_content_type', 'name': 'tests.test_strict_content_type_nested.test_strict_inner_on_lax_app_rejects_no_content_type'}
{'id': 'tests.test_strict_content_type_nested.test_default_inner_inherits_lax_from_app', 'name': 'tests.test_strict_content_type_nested.test_default_inner_inherits_lax_from_app'}
{'id': 'tests.test_strict_content_type_nested.test_strict_inner_accepts_json_content_type', 'name': 'tests.test_strict_content_type_nested.test_strict_inner_accepts_json_content_type'}
{'id': 'tests.test_strict_content_type_nested.test_default_inner_accepts_json_content_type', 'name': 'tests.test_strict_content_type_nested.test_default_inner_accepts_json_content_type'}
{'id': 'tests.test_strict_content_type_nested.test_lax_outer_on_strict_app_accepts_no_content_type', 'name': 'tests.test_strict_content_type_nested.t

In [60]:
# Inspect graph neighbors around a known code symbol

row = tasks_df[
    tasks_df["instance_id"] == "fastapi_14964"
].iloc[0]

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

target = "fastapi.responses.ORJSONResponse"

# Build adjacency lists
incoming = {}
outgoing = {}

for edge in graph["edges"]:
    source = edge.get("source")
    target_node = edge.get("target")

    outgoing.setdefault(source, []).append(target_node)
    incoming.setdefault(target_node, []).append(source)

print("Target:", target)

print("\nOutgoing neighbors:")
for node in outgoing.get(target, [])[:20]:
    print(node)

print("\nIncoming neighbors:")
for node in incoming.get(target, [])[:20]:
    print(node)

print("\nCounts:")
print("Outgoing:", len(outgoing.get(target, [])))
print("Incoming:", len(incoming.get(target, [])))

Target: fastapi.responses.ORJSONResponse

Outgoing neighbors:
fastapi.responses.ORJSONResponse.render

Incoming neighbors:
fastapi.responses.ORJSONResponse.render

Counts:
Outgoing: 1
Incoming: 1


In [61]:
# GraphFix: build graph adjacency and node metadata

from collections import defaultdict, deque
import re
import math


def build_graph_index(graph):
    """
    Build efficient node and adjacency indexes.
    """

    nodes_by_id = {}
    outgoing = defaultdict(list)
    incoming = defaultdict(list)

    for node in graph["nodes"]:
        node_id = str(node.get("id", ""))
        nodes_by_id[node_id] = node

    for edge in graph["edges"]:
        source = str(edge.get("source"))
        target = str(edge.get("target"))

        outgoing[source].append(target)
        incoming[target].append(source)

    return nodes_by_id, outgoing, incoming


def graph_distance(start_nodes, outgoing, incoming, max_depth=2):
    """
    Compute shortest undirected graph distance from any
    anchor node.

    We use both incoming and outgoing call edges so that
    callers and callees can contribute context.
    """

    distances = {}

    queue = deque()

    for node in start_nodes:
        if node not in distances:
            distances[node] = 0
            queue.append(node)

    while queue:

        current = queue.popleft()
        depth = distances[current]

        if depth >= max_depth:
            continue

        neighbors = set(
            outgoing.get(current, [])
            + incoming.get(current, [])
        )

        for neighbor in neighbors:

            if neighbor not in distances:
                distances[neighbor] = depth + 1
                queue.append(neighbor)

    return distances


def node_file_path(node_id):
    """
    Try to recover a repository file path from a graph symbol.
    """

    if not isinstance(node_id, str):
        return None

    # Most graph symbols use dotted module names.
    parts = node_id.split(".")

    # Convert obvious Python module portions to paths.
    if len(parts) >= 2:
        for i, part in enumerate(parts):
            if part in {"tests", "fastapi", "docs_src", "scripts"}:
                return "/".join(parts[i:-1])

    return None


print("GraphFix graph infrastructure loaded.")

GraphFix graph infrastructure loaded.


In [63]:
# Test GraphFix graph infrastructure

row = tasks_df[
    tasks_df["instance_id"] == "fastapi_14964"
].iloc[0]

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

nodes_by_id, outgoing, incoming = build_graph_index(graph)

anchor = "fastapi.responses.ORJSONResponse"

distances = graph_distance(
    [anchor],
    outgoing,
    incoming,
    max_depth=2
)

print("Anchor:", anchor)
print("Total nodes in graph:", len(nodes_by_id))
print("Nodes reachable within 2 hops:", len(distances))

print("\nNearest nodes:")

for node_id, distance in sorted(
    distances.items(),
    key=lambda x: (x[1], x[0])
)[:30]:

    print(f"distance={distance} | {node_id}")

Anchor: fastapi.responses.ORJSONResponse
Total nodes in graph: 4115
Nodes reachable within 2 hops: 2

Nearest nodes:
distance=0 | fastapi.responses.ORJSONResponse
distance=1 | fastapi.responses.ORJSONResponse.render


In [64]:
# Measure graph connectivity for issue-derived symbol anchors

test_instances = [
    "fastapi_14964",
    "fastapi_14978",
    "fastapi_15589",
    "fastapi_15280"
]

for instance_id in test_instances:

    row = tasks_df[
        tasks_df["instance_id"] == instance_id
    ].iloc[0]

    graph, embeddings = load_resources(
        row["repo"],
        row["base_commit"]
    )

    nodes_by_id, outgoing, incoming = build_graph_index(graph)

    candidates = extract_code_candidates(
        row["problem_statement"]
    )

    # Find metadata matches only
    anchors = []

    for query in candidates:

        for node in graph["nodes"]:

            node_id = str(node.get("id", ""))
            node_name = str(node.get("name", ""))

            if (
                query.lower() in node_id.lower()
                or query.lower() in node_name.lower()
            ):
                anchors.append(node_id)

    anchors = list(dict.fromkeys(anchors))

    print("\n" + "=" * 90)
    print("INSTANCE:", instance_id)
    print("Changed files:", row["changed_files"])
    print("Candidate terms:", candidates)
    print("Anchor nodes found:", len(anchors))

    for anchor in anchors[:10]:

        distances = graph_distance(
            [anchor],
            outgoing,
            incoming,
            max_depth=2
        )

        print(
            f"\nAnchor: {anchor}"
            f"\n  1-hop: {sum(d == 1 for d in distances.values())}"
            f"\n  2-hop total: {len(distances)}"
        )


INSTANCE: fastapi_14964
Changed files: ['fastapi/responses.py']
Candidate terms: ['Deprecate', 'Now', 'ORJSONResponse', 'UJSONResponse', 'github.com']
Anchor nodes found: 21

Anchor: tests.test_regex_deprecated_params.get_client
  1-hop: 5
  2-hop total: 122

Anchor: tests.test_regex_deprecated_params.test_query_params_str_validations_no_query
  1-hop: 1
  2-hop total: 6

Anchor: tests.test_regex_deprecated_params.test_query_params_str_validations_q_fixedquery
  1-hop: 1
  2-hop total: 6

Anchor: tests.test_regex_deprecated_params.test_query_params_str_validations_item_query_nonregexquery
  1-hop: 1
  2-hop total: 6

Anchor: tests.test_regex_deprecated_params.test_openapi_schema
  1-hop: 1
  2-hop total: 6

Anchor: tests.test_deprecated_openapi_prefix.read_main
  1-hop: 0
  2-hop total: 1

Anchor: tests.test_deprecated_openapi_prefix.test_main
  1-hop: 0
  2-hop total: 1

Anchor: tests.test_deprecated_openapi_prefix.test_openapi
  1-hop: 0
  2-hop total: 1

Anchor: tests.test_regex_de

In [5]:
# GraphFix: robust code-anchor discovery

import re
import numpy as np
from collections import defaultdict


STOPWORDS = {
    "the", "this", "that", "with", "from", "into", "when", "where",
    "which", "would", "could", "should", "have", "has", "been", "were",
    "will", "does", "doesn", "not", "only", "also", "than", "then",
    "there", "their", "they", "them", "you", "your", "for", "and",
    "but", "are", "was", "can", "may", "use", "using", "used",
    "need", "needs", "new", "now", "once", "make", "made", "support",
    "added", "add", "fix", "issue", "problem", "change"
}


def extract_anchor_terms(text):
    """
    Extract code-like terms from an issue description.

    We prefer:
    - snake_case identifiers
    - dotted identifiers
    - CamelCase names
    - ALL_CAPS names
    - longer distinctive tokens
    """

    if not isinstance(text, str):
        return []

    patterns = [
        r'\b[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_.]*\b',
        r'\b[A-Za-z_][A-Za-z0-9_]*_[A-Za-z0-9_]+\b',
        r'\b[A-Z][A-Za-z0-9]+(?:[A-Z][A-Za-z0-9]+)+\b',
        r'\b[A-Z]{2,}[A-Za-z0-9_]*\b',
    ]

    terms = []

    for pattern in patterns:
        terms.extend(re.findall(pattern, text))

    # Also keep unusually long normal words.
    words = re.findall(r'\b[A-Za-z][A-Za-z0-9_-]{7,}\b', text)

    terms.extend(words)

    cleaned = []

    for term in terms:
        term = term.strip(".,:;()[]{}<>\"'`")

        if len(term) < 5:
            continue

        if term.lower() in STOPWORDS:
            continue

        cleaned.append(term)

    # Preserve order while removing duplicates
    return list(dict.fromkeys(cleaned))


def find_anchor_nodes(graph, terms, max_per_term=10):
    """
    Find graph nodes that are plausible code anchors.

    Priority:
    1. Exact metadata match
    2. Exact source-code token match
    """

    nodes = graph["nodes"]

    metadata_matches = []
    source_matches = []

    for term in terms:

        term_lower = term.lower()

        # --------------------------------------------------
        # 1. Metadata matching
        # --------------------------------------------------

        for node in nodes:

            node_id = str(node.get("id", ""))
            node_name = str(node.get("name", ""))

            if (
                node_id.lower() == term_lower
                or node_name.lower() == term_lower
                or node_id.lower().endswith("." + term_lower)
                or node_name.lower().endswith("." + term_lower)
            ):
                metadata_matches.append(
                    (term, node_id, 3.0)
                )

        # --------------------------------------------------
        # 2. Source-code matching
        # --------------------------------------------------

        token_pattern = r'(?<![A-Za-z0-9_])' + re.escape(term) + r'(?![A-Za-z0-9_])'

        for node in nodes:

            node_id = str(node.get("id", ""))
            source = str(node.get("text", ""))

            if re.search(token_pattern, source, flags=re.IGNORECASE):

                source_matches.append(
                    (term, node_id, 1.0)
                )

    # Combine scores
    scores = defaultdict(float)
    matched_terms = defaultdict(list)

    for term, node_id, score in metadata_matches:
        scores[node_id] += score
        matched_terms[node_id].append(term)

    for term, node_id, score in source_matches:
        scores[node_id] += score
        matched_terms[node_id].append(term)

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for node_id, score in ranked:

        results.append({
            "node": node_id,
            "score": score,
            "matched_terms": list(dict.fromkeys(matched_terms[node_id]))
        })

    return results


# ----------------------------------------------------------
# Test the improved anchor discovery
# ----------------------------------------------------------

test_instances = [
    "fastapi_14964",
    "fastapi_14978",
    "fastapi_15589",
    "fastapi_15280"
]

for instance_id in test_instances:

    row = tasks_df[
        tasks_df["instance_id"] == instance_id
    ].iloc[0]

    graph, embeddings = load_resources(
        row["repo"],
        row["base_commit"]
    )

    terms = extract_anchor_terms(
        row["problem_statement"]
    )

    anchors = find_anchor_nodes(
        graph,
        terms
    )

    print("\n" + "=" * 90)
    print("INSTANCE:", instance_id)
    print("Changed files:", row["changed_files"])
    print("Anchor terms:", terms[:20])

    print("\nTop anchor nodes:")

    for item in anchors[:15]:

        print(
            f"{item['score']:5.1f} | "
            f"{item['node']} | "
            f"{item['matched_terms']}"
        )

NameError: name 'tasks_df' is not defined

In [79]:
# ============================================================
# GraphFix Stage 10: Symbol-seeded embedding retrieval
# ============================================================

def cosine_similarity_matrix(query_vector, matrix):
    """
    Compute cosine similarity between one query vector
    and all rows in matrix.
    """
    
    query_norm = np.linalg.norm(query_vector)

    if query_norm == 0:
        return np.zeros(len(matrix))

    matrix_norms = np.linalg.norm(matrix, axis=1)

    denominator = matrix_norms * query_norm

    similarities = np.zeros(len(matrix))

    valid = denominator > 0

    similarities[valid] = (
        matrix[valid] @ query_vector
    ) / denominator[valid]

    return similarities


def embedding_retrieve(graph, embeddings, anchor_nodes, top_k=20):
    """
    Retrieve graph nodes using embeddings associated with
    discovered code-symbol anchors.

    Multiple anchors are supported.
    For each candidate node we retain the strongest
    similarity to any anchor.
    """

    node_ids = [
        str(node["id"])
        for node in graph["nodes"]
    ]

    # --------------------------------------------------------
    # Build embedding matrix
    # --------------------------------------------------------

    vectors = []

    valid_ids = []

    for node_id in node_ids:

        if node_id in embeddings:

            vector = np.asarray(
                embeddings[node_id],
                dtype=np.float32
            )

            if vector.ndim == 1:
                vectors.append(vector)
                valid_ids.append(node_id)

    if not vectors:
        return []

    matrix = np.vstack(vectors)

    # --------------------------------------------------------
    # Query using every discovered anchor
    # --------------------------------------------------------

    best_scores = defaultdict(float)
    best_anchor = {}

    for anchor in anchor_nodes:

        if anchor not in embeddings:
            continue

        query_vector = np.asarray(
            embeddings[anchor],
            dtype=np.float32
        )

        similarities = cosine_similarity_matrix(
            query_vector,
            matrix
        )

        for node_id, similarity in zip(
            valid_ids,
            similarities
        ):

            similarity = float(similarity)

            if similarity > best_scores[node_id]:

                best_scores[node_id] = similarity
                best_anchor[node_id] = anchor

    ranked = sorted(
        best_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for node_id, score in ranked[:top_k]:

        results.append({
            "node": node_id,
            "similarity": score,
            "anchor": best_anchor[node_id]
        })

    return results

In [75]:
# ============================================================
# Test symbol-seeded embedding retrieval
# ============================================================

test_instances = [
    "fastapi_14964",
    "fastapi_15589"
]

for instance_id in test_instances:

    row = tasks_df[
        tasks_df["instance_id"] == instance_id
    ].iloc[0]

    graph, embeddings = load_resources(
        row["repo"],
        row["base_commit"]
    )

    terms = extract_anchor_terms(
        row["problem_statement"]
    )

    anchor_results = find_anchor_nodes(
        graph,
        terms
    )

    # Keep only the strongest anchors
    anchor_nodes = [
        item["node"]
        for item in anchor_results[:5]
        if item["node"] in embeddings
    ]

    retrieval = embedding_retrieve(
        graph,
        embeddings,
        anchor_nodes,
        top_k=15
    )

    print("\n" + "=" * 90)
    print("INSTANCE:", instance_id)
    print("Changed files:", row["changed_files"])
    print("Anchors:", anchor_nodes)

    print("\nEmbedding retrieval:")

    for item in retrieval:

        print(
            f"{item['similarity']:.4f} | "
            f"{item['node']} | "
            f"anchor={item['anchor']}"
        )


INSTANCE: fastapi_14964
Changed files: ['fastapi/responses.py']
Anchors: ['fastapi.responses.ORJSONResponse', 'fastapi.responses.UJSONResponse', 'tests.test_default_response_class.ORJSONResponse', 'fastapi.openapi.models.Response', 'fastapi.routing.APIRouter.__init__']

Embedding retrieval:
1.0000 | fastapi.routing.APIRouter.__init__ | anchor=fastapi.routing.APIRouter.__init__
1.0000 | tests.test_default_response_class.ORJSONResponse.render | anchor=tests.test_default_response_class.ORJSONResponse
1.0000 | fastapi.responses.UJSONResponse.render | anchor=fastapi.responses.UJSONResponse
1.0000 | fastapi.responses.ORJSONResponse | anchor=fastapi.responses.ORJSONResponse
1.0000 | fastapi.responses.ORJSONResponse.render | anchor=fastapi.responses.ORJSONResponse
1.0000 | fastapi.openapi.models.Response | anchor=fastapi.openapi.models.Response
1.0000 | fastapi.responses.UJSONResponse | anchor=fastapi.responses.UJSONResponse
1.0000 | tests.test_default_response_class.ORJSONResponse | anchor=t

In [118]:
# ============================================================
# GraphFix: Adaptive Graph-Guided Context Selection
# ============================================================

import math


def normalized_embedding_scores(retrieval):
    """
    Convert raw cosine similarities into a relative score.

    Because many supplied embeddings have very high cosine
    similarity, we normalize relative to the retrieved range.
    """

    if not retrieval:
        return {}

    scores = [
        item["similarity"]
        for item in retrieval
    ]

    minimum = min(scores)
    maximum = max(scores)

    if maximum - minimum < 1e-8:
        return {
            item["node"]: 1.0
            for item in retrieval
        }

    return {
        item["node"]:
        (item["similarity"] - minimum) /
        (maximum - minimum)
        for item in retrieval
    }


def graphfix_expand(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    max_depth=3,
    alpha=0.55,
    beta=0.30,
    gamma=0.15
):
    """
    Adaptive graph-guided context selection.

    Score components:

    - semantic_score:
        supplied embedding similarity

    - structural_score:
        proximity to discovered anchors

    - diversity_score:
        preference for nodes not already represented
        by the selected neighborhood

    The selector greedily chooses high-value nodes until
    the context budget is reached.
    """

    nodes_by_id, outgoing, incoming = build_graph_index(graph)

    # --------------------------------------------------------
    # Graph distances from all anchors
    # --------------------------------------------------------

    distances = graph_distance(
        anchor_nodes,
        outgoing,
        incoming,
        max_depth=max_depth
    )

    # --------------------------------------------------------
    # Candidate pool
    # --------------------------------------------------------

    candidates = set(embedding_scores.keys())

    # Add graph neighbors around anchors
    frontier = set(anchor_nodes)

    for _ in range(max_depth):

        next_frontier = set()

        for node in frontier:

            next_frontier.update(
                outgoing.get(node, [])
            )

            next_frontier.update(
                incoming.get(node, [])
            )

        candidates.update(next_frontier)
        frontier = next_frontier

    # Keep only actual graph nodes
    candidates = {
        node
        for node in candidates
        if node in nodes_by_id
    }

    # --------------------------------------------------------
    # Score candidates
    # --------------------------------------------------------

    scored = []

    for node in candidates:

        # Embedding relevance
        semantic_score = embedding_scores.get(
            node,
            0.0
        )

        # Structural proximity
        distance = distances.get(
            node,
            max_depth + 1
        )

        if distance <= max_depth:
            structural_score = 1.0 / (1.0 + distance)
        else:
            structural_score = 0.0

        # Initial diversity score
        diversity_score = 1.0

        total_score = (
            alpha * semantic_score
            + beta * structural_score
            + gamma * diversity_score
        )

        scored.append({
            "node": node,
            "score": total_score,
            "semantic_score": semantic_score,
            "structural_score": structural_score,
            "distance": distance
        })

    scored.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    # --------------------------------------------------------
    # Adaptive selection
    # --------------------------------------------------------

    selected = []
    selected_modules = set()

    for item in scored:

        node = item["node"]

        # Derive a coarse module prefix
        parts = node.split(".")

        if len(parts) >= 2:
            module = ".".join(parts[:2])
        else:
            module = node

        # Penalize excessive repetition from the same module
        repetition_penalty = (
            0.85
            if module in selected_modules
            else 1.0
        )

        adjusted_score = (
            item["score"] * repetition_penalty
        )

        item["adjusted_score"] = adjusted_score

        if adjusted_score <= 0:
            continue

        selected.append(item)
        selected_modules.add(module)

        if len(selected) >= budget:
            break

    return selected

In [94]:
# ============================================================
# Test GraphFix on the inspected tasks
# ============================================================

test_instances = [
    "fastapi_14964",
    "fastapi_15589"
]

for instance_id in test_instances:

    row = tasks_df[
        tasks_df["instance_id"] == instance_id
    ].iloc[0]

    graph, embeddings = load_resources(
        row["repo"],
        row["base_commit"]
    )

    # --------------------------------------------------------
    # Discover anchors
    # --------------------------------------------------------

    terms = extract_anchor_terms(
        row["problem_statement"]
    )

    anchor_results = find_anchor_nodes(
        graph,
        terms
    )

    anchor_nodes = [
        item["node"]
        for item in anchor_results[:5]
        if item["node"] in embeddings
    ]

    # --------------------------------------------------------
    # Embedding retrieval
    # --------------------------------------------------------

    retrieval = embedding_retrieve(
        graph,
        embeddings,
        anchor_nodes,
        top_k=30
    )

    embedding_scores = normalized_embedding_scores(
        retrieval
    )

    # --------------------------------------------------------
    # GraphFix
    # --------------------------------------------------------

    selected = graphfix_expand(
        graph,
        anchor_nodes,
        embedding_scores,
        budget=30,
        max_depth=3
    )

    print("\n" + "=" * 90)
    print("INSTANCE:", instance_id)
    print("Changed files:", row["changed_files"])
    print("Anchors:", anchor_nodes)

    print("\nGraphFix selected nodes:")

    for item in selected:

        print(
            f"{item['adjusted_score']:.4f} | "
            f"distance={item['distance']} | "
            f"semantic={item['semantic_score']:.3f} | "
            f"structural={item['structural_score']:.3f} | "
            f"{item['node']}"
        )


INSTANCE: fastapi_14964
Changed files: ['fastapi/responses.py']
Anchors: ['fastapi.responses.ORJSONResponse', 'fastapi.responses.UJSONResponse', 'tests.test_default_response_class.ORJSONResponse', 'fastapi.openapi.models.Response', 'fastapi.routing.APIRouter.__init__']

GraphFix selected nodes:
1.0000 | distance=0 | semantic=1.000 | structural=1.000 | fastapi.routing.APIRouter.__init__
1.0000 | distance=0 | semantic=1.000 | structural=1.000 | fastapi.openapi.models.Response
1.0000 | distance=0 | semantic=1.000 | structural=1.000 | fastapi.responses.ORJSONResponse
0.8500 | distance=0 | semantic=1.000 | structural=1.000 | fastapi.responses.UJSONResponse
1.0000 | distance=0 | semantic=1.000 | structural=1.000 | tests.test_default_response_class.ORJSONResponse
0.7225 | distance=1 | semantic=1.000 | structural=0.500 | tests.test_default_response_class.ORJSONResponse.render
0.7225 | distance=1 | semantic=1.000 | structural=0.500 | fastapi.responses.ORJSONResponse.render
0.7225 | distance=1 

In [141]:
# ============================================================
# GraphFix evaluation helpers
# ============================================================

def file_to_module_prefix(path):
    """
    Convert a Python file path into the module prefix used
    by graph node identifiers.
    """
    
    path = path.replace("\\", "/")
    
    if path.startswith("./"):
        path = path[2:]
    
    if path.endswith(".py"):
        path = path[:-3]
    
    return path.replace("/", ".")


def node_matches_file(node_id, file_path):
    """
    Approximate whether a graph node belongs to a source file
    using the module prefix encoded in the node identifier.
    """
    
    prefix = file_to_module_prefix(file_path)

    return (
        node_id == prefix
        or node_id.startswith(prefix + ".")
    )


def changed_file_hits(selected_nodes, changed_files):
    """
    Determine which reference-patch files are represented
    by the selected graph nodes.
    """
    
    hits = set()

    for node in selected_nodes:

        for file_path in changed_files:

            if node_matches_file(node, file_path):
                hits.add(file_path)

    return hits


def localization_metrics(
    selected_nodes,
    changed_files
):
    """
    Compute file-level localization metrics.
    """

    changed_files = list(dict.fromkeys(changed_files))

    if not changed_files:
        return {
            "coverage": 0.0,
            "precision": 0.0,
            "context_size": len(selected_nodes),
            "hits": 0
        }

    hits = changed_file_hits(
        selected_nodes,
        changed_files
    )

    coverage = (
        len(hits) / len(changed_files)
    )

    relevant_nodes = 0

    for node in selected_nodes:

        if any(
            node_matches_file(node, f)
            for f in changed_files
        ):
            relevant_nodes += 1

    precision = (
        relevant_nodes / len(selected_nodes)
        if selected_nodes
        else 0.0
    )

    return {
        "coverage": coverage,
        "precision": precision,
        "context_size": len(selected_nodes),
        "hits": len(hits)
    }

In [143]:
# ============================================================
# FIX: Baseline functions
# ============================================================

def baseline_anchor_retrieval(
    graph,
    anchor_results,
    budget=30
):
    return [
        item["node"]
        for item in anchor_results[:budget]
    ]


def baseline_embedding_retrieval(
    graph,
    embeddings,
    anchor_nodes,
    budget=30
):
    retrieval = embedding_retrieve(
        graph,
        embeddings,
        anchor_nodes,
        top_k=budget
    )

    return [
        item["node"]
        for item in retrieval
    ]


def baseline_fixed_graph(
    graph,
    anchor_nodes,
    depth=2,
    budget=30
):
    nodes_by_id, outgoing, incoming = build_graph_index(
        graph
    )

    distances = graph_distance(
        anchor_nodes,
        outgoing,
        incoming,
        max_depth=depth
    )

    ranked = sorted(
        distances.items(),
        key=lambda x: (x[1], x[0])
    )

    return [
        node
        for node, distance in ranked[:budget]
    ]

print("Baseline functions loaded successfully.")

Baseline functions loaded successfully.


In [147]:
# ============================================================
# Run the experiment across all unique task snapshots
# ============================================================

results = []

unique_tasks = (
    tasks_df
    .drop_duplicates(
        subset=["repo", "base_commit"]
    )
    .copy()
)

print(
    "Unique task snapshots:",
    len(unique_tasks)
)

for idx, (_, row) in enumerate(
    unique_tasks.iterrows()
):

    try:

        graph, embeddings = load_resources(
            row["repo"],
            row["base_commit"]
        )

        terms = extract_anchor_terms(
            row["problem_statement"]
        )

        anchor_results = find_anchor_nodes(
            graph,
            terms
        )

        anchor_nodes = [
            item["node"]
            for item in anchor_results[:5]
            if item["node"] in embeddings
        ]

        retrieval = embedding_retrieve(
            graph,
            embeddings,
            anchor_nodes,
            top_k=30
        )

        embedding_scores = normalized_embedding_scores(
            retrieval
        )

        anchor_nodes_selected = baseline_anchor_retrieval(
            graph,
            anchor_results,
            budget=30
        )

        embedding_nodes_selected = baseline_embedding_retrieval(
            graph,
            embeddings,
            anchor_nodes,
            budget=30
        )

        fixed_graph_nodes = baseline_fixed_graph(
            graph,
            anchor_nodes,
            depth=2,
            budget=30
        )

        graphfix_selected = graphfix_expand(
            graph,
            anchor_nodes,
            embedding_scores,
            budget=30,
            max_depth=3
        )

        graphfix_nodes = [
            item["node"]
            for item in graphfix_selected
        ]

        methods = {
            "anchor": anchor_nodes_selected,
            "embedding": embedding_nodes_selected,
            "fixed_graph": fixed_graph_nodes,
            "graphfix": graphfix_nodes
        }

        for method, selected_nodes in methods.items():

            metrics = localization_metrics(
                selected_nodes,
                row["changed_files"]
            )

            results.append({
                "instance_id": row["instance_id"],
                "repo": row["repo"],
                "base_commit": row["base_commit"],
                "method": method,
                "coverage": metrics["coverage"],
                "precision": metrics["precision"],
                "context_size": metrics["context_size"],
                "hits": metrics["hits"],
                "num_anchors": len(anchor_nodes),
                "num_anchor_candidates": len(anchor_results)
            })

        if (idx + 1) % 10 == 0:
            print(
                f"Processed {idx + 1}/{len(unique_tasks)}"
            )

    except Exception as e:

        print(
            "ERROR:",
            row["instance_id"],
            str(e)
        )

results_df = pd.DataFrame(results)

print("\nExperiment complete.")
print("Rows:", len(results_df))

results_df.head()

Unique task snapshots: 127
Processed 10/127
Processed 20/127
Processed 30/127
Processed 40/127
Processed 50/127
Processed 60/127
Processed 70/127
Processed 80/127
Processed 90/127
Processed 100/127
Processed 110/127
Processed 120/127

Experiment complete.
Rows: 508


,instance_id,repo,base_commit,method,coverage,precision,context_size,hits,num_anchors,num_anchor_candidates
0,fastapi_15661,fastapi/fastapi,ee22a4b8ca46dcce26c8c183afc4992a888d8be2,anchor,0.0,0.0,30,0,5,512
1,fastapi_15661,fastapi/fastapi,ee22a4b8ca46dcce26c8c183afc4992a888d8be2,embedding,0.0,0.0,30,0,5,512
2,fastapi_15661,fastapi/fastapi,ee22a4b8ca46dcce26c8c183afc4992a888d8be2,fixed_graph,0.0,0.0,30,0,5,512
3,fastapi_15661,fastapi/fastapi,ee22a4b8ca46dcce26c8c183afc4992a888d8be2,graphfix,0.0,0.0,30,0,5,512
4,fastapi_15588,fastapi/fastapi,cb83b83dcf78eab4ea17d504db5abcda705fbdc4,anchor,0.0,0.0,30,0,5,721


In [155]:
summary = (
    results_df
    .groupby("method")
    .agg(
        mean_coverage=("coverage", "mean"),
        mean_precision=("precision", "mean"),
        mean_context_size=("context_size", "mean"),
        mean_anchors=("num_anchors", "mean"),
        tasks=("instance_id", "nunique")
    )
    .reset_index()
)

summary

,method,mean_coverage,mean_precision,mean_context_size,mean_anchors,tasks
0,anchor,0.417668,0.107446,27.645669,4.88189,127
1,embedding,0.407259,0.087927,29.763780,4.88189,127
2,fixed_graph,0.407106,0.105407,29.023622,4.88189,127
3,graphfix,0.414127,0.093438,29.763780,4.88189,127


In [156]:
# How many changed files are Python files?

all_changed = tasks_df["changed_files"].explode().dropna()

python_files = all_changed[
    all_changed.str.endswith(".py")
]

print("Total changed files:", len(all_changed))
print("Python changed files:", len(python_files))
print(
    "Python percentage:",
    round(len(python_files) / len(all_changed) * 100, 2),
    "%"
)

Total changed files: 278
Python changed files: 278
Python percentage: 100.0 %


In [157]:
anchor_availability = (
    results_df.groupby("instance_id")["num_anchors"]
    .first()
)

print("Total tasks:", len(anchor_availability))
print("Tasks with anchors:", (anchor_availability > 0).sum())
print("Tasks with no anchors:", (anchor_availability == 0).sum())
print(
    "Anchor availability:",
    round((anchor_availability > 0).mean() * 100, 2),
    "%"
)

Total tasks: 127
Tasks with anchors: 126
Tasks with no anchors: 1
Anchor availability: 99.21 %


In [158]:

coverage_pivot = results_df.pivot(
    index="instance_id",
    columns="method",
    values="coverage"
)

coverage_pivot["graphfix_minus_anchor"] = (
    coverage_pivot["graphfix"] - coverage_pivot["anchor"]
)

coverage_pivot["graphfix_minus_embedding"] = (
    coverage_pivot["graphfix"] - coverage_pivot["embedding"]
)

coverage_pivot["graphfix_minus_fixed_graph"] = (
    coverage_pivot["graphfix"] - coverage_pivot["fixed_graph"]
)

coverage_pivot[
    [
        "graphfix_minus_anchor",
        "graphfix_minus_embedding",
        "graphfix_minus_fixed_graph"
    ]
].mean()

method
graphfix_minus_anchor        -0.003540
graphfix_minus_embedding      0.006868
graphfix_minus_fixed_graph    0.007021
dtype: float64

In [159]:
for baseline in ["anchor", "embedding", "fixed_graph"]:
    delta = coverage_pivot["graphfix"] - coverage_pivot[baseline]

    print("\nGraphFix vs", baseline)
    print("Improved:", (delta > 0).sum())
    print("Same:    ", (delta == 0).sum())
    print("Worse:   ", (delta < 0).sum())
    print("Mean Δ:  ", round(delta.mean(), 4))
    print("Median Δ:", round(delta.median(), 4))


GraphFix vs anchor
Improved: 12
Same:     98
Worse:    17
Mean Δ:   -0.0035
Median Δ: 0.0

GraphFix vs embedding
Improved: 6
Same:     112
Worse:    9
Mean Δ:   0.0069
Median Δ: 0.0

GraphFix vs fixed_graph
Improved: 8
Same:     112
Worse:    7
Mean Δ:   0.007
Median Δ: 0.0


In [163]:
# Compare how often each method has exactly the same localization outcome

coverage_wide = results_df.pivot(
    index="instance_id",
    columns="method",
    values="coverage"
)

print("Coverage patterns:")
print(
    coverage_wide[
        ["anchor", "embedding", "fixed_graph", "graphfix"]
    ].value_counts().head(20)
)

Coverage patterns:
anchor    embedding  fixed_graph  graphfix
0.000000  0.000000   0.000000     0.000000    45
1.000000  1.000000   1.000000     1.000000    31
          0.000000   0.000000     0.000000     5
0.500000  0.500000   0.500000     0.500000     4
0.000000  0.000000   1.000000     1.000000     3
0.500000  0.000000   0.000000     0.000000     3
0.000000  1.000000   1.000000     1.000000     3
                     0.000000     1.000000     3
1.000000  0.000000   1.000000     1.000000     2
0.000000  0.250000   0.250000     0.250000     1
          0.000000   1.000000     0.000000     1
                     0.111111     0.000000     1
          1.000000   0.000000     0.000000     1
0.285714  0.571429   0.142857     0.571429     1
0.300000  0.100000   0.200000     0.100000     1
0.333333  0.333333   0.000000     0.000000     1
0.300000  0.300000   0.300000     0.300000     1
0.000000  1.000000   1.000000     0.000000     1
0.200000  0.300000   0.200000     0.200000     1
0.25000

In [165]:
# Number of unique coverage values produced by each method

for method in ["anchor", "embedding", "fixed_graph", "graphfix"]:
    values = results_df.loc[
        results_df["method"] == method,
        "coverage"
    ]

    print(
        method,
        "unique coverage values:",
        values.nunique()
    )

anchor unique coverage values: 12
embedding unique coverage values: 13
fixed_graph unique coverage values: 12
graphfix unique coverage values: 12


In [166]:
# Tasks where GraphFix differs from Anchor

comparison = results_df.pivot(
    index="instance_id",
    columns="method",
    values="coverage"
).reset_index()

comparison["graphfix_vs_anchor"] = (
    comparison["graphfix"] - comparison["anchor"]
)

comparison.sort_values(
    "graphfix_vs_anchor",
    ascending=False
).head(15)

method,instance_id,anchor,embedding,fixed_graph,graphfix,graphfix_vs_anchor
3,fastapi_13207,0.000000,1.000000,0.000000,1.000000,1.000000
28,fastapi_14455,0.000000,0.000000,1.000000,1.000000,1.000000
56,fastapi_15589,0.000000,1.000000,1.000000,1.000000,1.000000
124,rich_4076,0.000000,1.000000,0.000000,1.000000,1.000000
126,rich_4079,0.000000,1.000000,1.000000,1.000000,1.000000
123,rich_4075,0.000000,1.000000,1.000000,1.000000,1.000000
92,rich_3278,0.000000,1.000000,0.000000,1.000000,1.000000
100,rich_3480,0.000000,0.000000,1.000000,1.000000,1.000000
116,rich_3935,0.000000,0.000000,1.000000,1.000000,1.000000
105,rich_3535,0.500000,0.500000,1.000000,1.000000,0.500000


In [167]:
# Tasks where GraphFix loses most against Anchor

comparison.sort_values(
    "graphfix_vs_anchor",
    ascending=True
).head(15)

method,instance_id,anchor,embedding,fixed_graph,graphfix,graphfix_vs_anchor
21,fastapi_14360,1.000000,0.000000,0.0,0.00,-1.000000
26,fastapi_14430,1.000000,0.000000,0.0,0.00,-1.000000
19,fastapi_14349,1.000000,0.000000,0.0,0.00,-1.000000
34,fastapi_14485,1.000000,0.000000,0.0,0.00,-1.000000
41,fastapi_14616,1.000000,1.000000,0.0,0.00,-1.000000
93,rich_3296,1.000000,0.000000,0.0,0.00,-1.000000
85,rich_3061,1.000000,0.000000,1.0,0.00,-1.000000
18,fastapi_14306,1.000000,0.500000,0.5,0.50,-0.500000
107,rich_3676,0.500000,0.000000,0.0,0.00,-0.500000
91,rich_3180,0.500000,0.000000,0.0,0.00,-0.500000


In [168]:
for instance_id in ["fastapi_15589", "fastapi_14360"]:
    row = tasks_df[tasks_df["instance_id"] == instance_id].iloc[0]

    print("=" * 80)
    print("INSTANCE:", instance_id)
    print("REPO:", row["repo"])
    print("BASE COMMIT:", row["base_commit"])
    print("\nCHANGED FILES:")
    for f in row["changed_files"]:
        print(" -", f)

    print("\nPROBLEM STATEMENT:")
    print(row["problem_statement"])

INSTANCE: fastapi_15589
REPO: fastapi/fastapi
BASE COMMIT: 22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b

CHANGED FILES:
 - fastapi/dependencies/utils.py

PROBLEM STATEMENT:
♻️ Do not accept underscore headers when using `convert_underscores=True` (the default)

## Pull Request

<!--
Please start with a GitHub Discussion.

Once a team member asks you to open a PR, create it and link the discussion here.

Obvious typo fixes can be made in a PR without starting a discussion.
-->

Discussion: <!-- Link to the GitHub Discussion -->

## Description

♻️ Do not accept underscore headers when using `convert_underscores=True` (the default)

This means it would only accept header names separated with hyphens by default, and to accept header names separated with underscores, an app would need to explicitly set `convert_underscores=False`.

<!-- Write the description of your PR here -->

## AI Disclaimer

<!-- If using AI, write here the prompt and model used -->

<details>
<summary>AI transcript</summ

In [194]:
def graphfix_anchor_preserved(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    anchor_budget=5,
    max_depth=3,
    alpha=0.55,
    beta=0.30,
    gamma=0.15
):
    nodes_by_id, outgoing, incoming = build_graph_index(graph)

    # Keep valid anchors first
    preserved = []
    for node in anchor_nodes:
        if node in nodes_by_id and node not in preserved:
            preserved.append(node)

    preserved = preserved[:anchor_budget]

    remaining_budget = max(0, budget - len(preserved))

    # Graph distances from anchors
    distances = graph_distance(
        anchor_nodes,
        outgoing,
        incoming,
        max_depth=max_depth
    )

    # Candidate pool = embedding candidates + graph expansion
    candidates = set(embedding_scores.keys())

    frontier = set(anchor_nodes)

    for _ in range(max_depth):
        next_frontier = set()

        for node in frontier:
            next_frontier.update(outgoing.get(node, []))
            next_frontier.update(incoming.get(node, []))

        candidates.update(next_frontier)
        frontier = next_frontier

    candidates = {
        node for node in candidates
        if node in nodes_by_id and node not in preserved
    }

    scored = []

    for node in candidates:

        semantic_score = embedding_scores.get(node, 0.0)

        distance = distances.get(
            node,
            max_depth + 1
        )

        structural_score = (
            1 / (1 + distance)
            if distance <= max_depth
            else 0.0
        )

        diversity_score = 1.0

        total_score = (
            alpha * semantic_score
            + beta * structural_score
            + gamma * diversity_score
        )

        scored.append({
            "node": node,
            "score": total_score,
            "semantic_score": semantic_score,
            "distance": distance
        })

    scored.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    selected = list(preserved)
    selected_modules = set()

    for node in preserved:
        parts = node.split(".")
        module = ".".join(parts[:2]) if len(parts) >= 2 else node
        selected_modules.add(module)

    for item in scored:

        node = item["node"]

        parts = node.split(".")
        module = ".".join(parts[:2]) if len(parts) >= 2 else node

        repetition_penalty = (
            0.85
            if module in selected_modules
            else 1.0
        )

        adjusted_score = (
            item["score"] * repetition_penalty
        )

        item["adjusted_score"] = adjusted_score

    scored.sort(
        key=lambda x: x["adjusted_score"],
        reverse=True
    )

    for item in scored:

        if len(selected) >= budget:
            break

        node = item["node"]

        if node not in selected:
            selected.append(node)

    return selected

In [222]:
def find_anchor_nodes_fast(graph, terms, max_per_term=10):
    """
    Faster anchor discovery:
    1. Exact symbol/name matching first
    2. Simple substring matching in source text
    3. Avoids expensive regex scanning of every source string
    """

    terms = [str(t).strip() for t in terms if str(t).strip()]
    
    # Normalize terms once
    normalized_terms = []
    for term in terms:
        normalized_terms.append(term.lower())

    results = {}

    for node in graph["nodes"]:
        node_id = str(node.get("id", ""))
        name = str(node.get("name", ""))
        source = str(node.get("text", ""))

        node_id_lower = node_id.lower()
        name_lower = name.lower()

        matched = []
        score = 0

        for term, term_lower in zip(terms, normalized_terms):

            # Strong signal: exact symbol/name match
            if (
                node_id_lower == term_lower
                or name_lower == term_lower
                or node_id_lower.endswith("." + term_lower)
                or name_lower == term_lower
            ):
                matched.append(term)
                score += 3
                continue

            # Safer/faster source lookup
            if term_lower in source.lower():
                matched.append(term)
                score += 1

        if matched:
            results[node_id] = {
                "node": node_id,
                "score": score,
                "matched_terms": matched
            }

    ranked = sorted(
        results.values(),
        key=lambda x: (-x["score"], x["node"])
    )

    return ranked[:max_per_term * max(1, len(terms))]

In [197]:

for instance_id in ["fastapi_15589", "fastapi_14360"]:

    row = tasks_df[
        tasks_df["instance_id"] == instance_id
    ].iloc[0]

    graph, embeddings = load_resources(
        row["repo"],
        row["base_commit"]
    )

    terms = extract_anchor_terms(
        row["problem_statement"]
    )

    anchor_results = find_anchor_nodes_fast(
        graph,
        terms
    )

    anchor_nodes = [
        item["node"]
        for item in anchor_results[:5]
        if item["node"] in embeddings
    ]

    retrieval = embedding_retrieve(
        graph,
        embeddings,
        anchor_nodes,
        top_k=30
    )

    embedding_scores = normalized_embedding_scores(
        retrieval
    )

    selected = graphfix_anchor_preserved(
        graph,
        anchor_nodes,
        embedding_scores,
        budget=30,
        anchor_budget=5
    )

    metrics = localization_metrics(
        selected,
        row["changed_files"]
    )

    print("\n", "=" * 60)
    print(instance_id)
    print("Changed:", row["changed_files"])
    print("Terms:", terms)
    print("Anchors:", anchor_nodes)
    print("Coverage:", metrics["coverage"])
    print("Precision:", metrics["precision"])
    print("Hits:", metrics["hits"])


fastapi_15589
Changed: ['fastapi/dependencies/utils.py']
Terms: ['convert_underscores', 'GitHub', 'underscore', 'Discussion', 'discussion', 'starting', 'Description', 'separated', 'underscores', 'explicitly', 'description', 'Disclaimer', 'transcript', 'Checklist', 'proposed', 'Coverage', 'documentation', 'explains']
Anchors: ['fastapi.openapi.utils._get_openapi_operation_parameters', 'fastapi.param_functions.Header', 'fastapi.params.Header', 'fastapi.params.Header.__init__', 'fastapi.applications.FastAPI']
Coverage: 1.0
Precision: 0.1
Hits: 1

fastapi_14360
Changed: ['fastapi/dependencies/utils.py']
Terms: ['field.name', 'key.alias', 'processed_keys', 'parameter', 'currently', 'extracted', 'validate', 'searches', 'validation', 'correctly', 'mistakenly']
Anchors: ['fastapi._compat.v2.ModelField.validate', 'fastapi.applications.FastAPI', 'fastapi.dependencies.utils.request_params_to_args', 'fastapi._compat.model_field.ModelField.validate', 'fastapi.applications.FastAPI.delete']
Coverage

In [235]:
def baseline_anchor5(anchor_nodes, budget=30):
    return anchor_nodes[:min(len(anchor_nodes), budget)]

In [267]:
def prepare_embedding_matrix(graph, embeddings):
    node_ids = []
    vectors = []

    for node in graph["nodes"]:
        node_id = str(node["id"])

        if node_id in embeddings:
            vector = np.asarray(
                embeddings[node_id],
                dtype=np.float32
            )

            if vector.ndim == 1:
                node_ids.append(node_id)
                vectors.append(vector)

    matrix = np.vstack(vectors)

    return node_ids, matrix

In [280]:
def embedding_retrieve_fast(
    graph,
    embeddings,
    anchor_nodes,
    top_k=30
):
    node_ids, matrix = prepare_embedding_matrix(
        graph,
        embeddings
    )

    node_to_index = {
        node_id: i
        for i, node_id in enumerate(node_ids)
    }

    matrix_norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True
    )

    matrix_norms[matrix_norms == 0] = 1.0

    matrix_normalized = matrix / matrix_norms

    best_scores = {}
    best_anchor = {}

    for anchor in anchor_nodes:

        if anchor not in node_to_index:
            continue

        query = matrix[
            node_to_index[anchor]
        ]

        query_norm = np.linalg.norm(query)

        if query_norm == 0:
            continue

        query = query / query_norm

        similarities = matrix_normalized @ query

        top_indices = np.argsort(
            similarities
        )[-top_k:][::-1]

        for idx in top_indices:

            node_id = node_ids[idx]
            score = float(similarities[idx])

            if (
                node_id not in best_scores
                or score > best_scores[node_id]
            ):
                best_scores[node_id] = score
                best_anchor[node_id] = anchor

    results = []

    for node_id, score in best_scores.items():

        results.append({
            "node": node_id,
            "similarity": score,
            "anchor": best_anchor[node_id]
        })

    results.sort(
        key=lambda x: x["similarity"],
        reverse=True
    )

    return results[:top_k]

In [282]:
row = tasks_df[
    tasks_df["instance_id"] == "fastapi_15589"
].iloc[0]

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

terms = extract_anchor_terms(
    row["problem_statement"]
)

anchor_results = find_anchor_nodes_fast(
    graph,
    terms
)

anchor_nodes = [
    item["node"]
    for item in anchor_results[:5]
    if item["node"] in embeddings
]

print("Anchors:", anchor_nodes)

retrieval = embedding_retrieve_fast(
    graph,
    embeddings,
    anchor_nodes,
    top_k=30
)

print("Retrieved:", len(retrieval))
print(retrieval[:5])

Anchors: ['fastapi.openapi.utils._get_openapi_operation_parameters', 'fastapi.param_functions.Header', 'fastapi.params.Header', 'fastapi.params.Header.__init__', 'fastapi.applications.FastAPI']
Retrieved: 30
[{'node': 'fastapi.param_functions.Header', 'similarity': 1.0, 'anchor': 'fastapi.param_functions.Header'}, {'node': 'fastapi.params.Header.__init__', 'similarity': 1.0, 'anchor': 'fastapi.params.Header.__init__'}, {'node': 'fastapi.applications.FastAPI', 'similarity': 1.0, 'anchor': 'fastapi.applications.FastAPI'}, {'node': 'fastapi.openapi.utils._get_openapi_operation_parameters', 'similarity': 0.9999998807907104, 'anchor': 'fastapi.openapi.utils._get_openapi_operation_parameters'}, {'node': 'fastapi.params.Header', 'similarity': 0.9999998807907104, 'anchor': 'fastapi.params.Header'}]


In [283]:
embedding_scores = normalized_embedding_scores(
    retrieval
)

graphfix_selected = graphfix_anchor_preserved(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    anchor_budget=5
)

graphfix_metrics = localization_metrics(
    graphfix_selected,
    row["changed_files"]
)

print("Changed files:", row["changed_files"])
print("GraphFix-AP coverage:", graphfix_metrics["coverage"])
print("GraphFix-AP precision:", graphfix_metrics["precision"])
print("GraphFix-AP hits:", graphfix_metrics["hits"])
print("Selected nodes:", len(graphfix_selected))

Changed files: ['fastapi/dependencies/utils.py']
GraphFix-AP coverage: 1.0
GraphFix-AP precision: 0.1
GraphFix-AP hits: 1
Selected nodes: 30


In [285]:
row = tasks_df[
    tasks_df["instance_id"] == "fastapi_14360"
].iloc[0]

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

terms = extract_anchor_terms(
    row["problem_statement"]
)

anchor_results = find_anchor_nodes_fast(
    graph,
    terms
)

anchor_nodes = [
    item["node"]
    for item in anchor_results[:5]
    if item["node"] in embeddings
]

print("Anchors:", anchor_nodes)

anchor_selected = baseline_anchor5(
    anchor_nodes,
    budget=30
)

anchor_metrics = localization_metrics(
    anchor_selected,
    row["changed_files"]
)

retrieval = embedding_retrieve_fast(
    graph,
    embeddings,
    anchor_nodes,
    top_k=30
)

embedding_scores = normalized_embedding_scores(
    retrieval
)

graphfix_selected = graphfix_anchor_preserved(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    anchor_budget=5
)

graphfix_metrics = localization_metrics(
    graphfix_selected,
    row["changed_files"]
)

print("\nChanged files:", row["changed_files"])

print(
    "Anchor-5:",
    anchor_metrics["coverage"],
    anchor_metrics["precision"],
    anchor_metrics["hits"]
)

print(
    "GraphFix-AP:",
    graphfix_metrics["coverage"],
    graphfix_metrics["precision"],
    graphfix_metrics["hits"]
)

Anchors: ['fastapi._compat.v2.ModelField.validate', 'fastapi.applications.FastAPI', 'fastapi.dependencies.utils.request_params_to_args', 'fastapi._compat.model_field.ModelField.validate', 'fastapi.applications.FastAPI.delete']

Changed files: ['fastapi/dependencies/utils.py']
Anchor-5: 1.0 0.2 1
GraphFix-AP: 1.0 0.03333333333333333 1


In [286]:
graphfix_ap_rows = []

unique_tasks = tasks_df.drop_duplicates(
    subset=["repo", "base_commit"]
).copy()

print("Tasks to evaluate:", len(unique_tasks))

for i, (_, row) in enumerate(unique_tasks.iterrows(), start=1):

    graph, embeddings = load_resources(
        row["repo"],
        row["base_commit"]
    )

    # 1. Find issue-derived anchors
    terms = extract_anchor_terms(
        row["problem_statement"]
    )

    anchor_results = find_anchor_nodes_fast(
        graph,
        terms
    )

    anchor_nodes = [
        item["node"]
        for item in anchor_results[:5]
        if item["node"] in embeddings
    ]

    # Skip only if no usable anchors exist
    if not anchor_nodes:
        graphfix_ap_rows.append({
            "instance_id": row["instance_id"],
            "repo": row["repo"],
            "base_commit": row["base_commit"],
            "method": "graphfix_ap",
            "coverage": 0.0,
            "precision": 0.0,
            "context_size": 0,
            "num_anchors": 0,
            "hits": []
        })
        continue

    # 2. Fast embedding retrieval
    retrieval = embedding_retrieve_fast(
        graph,
        embeddings,
        anchor_nodes,
        top_k=30
    )

    embedding_scores = normalized_embedding_scores(
        retrieval
    )

    # 3. Anchor-preserved GraphFix
    selected = graphfix_anchor_preserved(
        graph,
        anchor_nodes,
        embedding_scores,
        budget=30,
        anchor_budget=5
    )

    # 4. Evaluate localization
    metrics = localization_metrics(
        selected,
        row["changed_files"]
    )

    graphfix_ap_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "base_commit": row["base_commit"],
        "method": "graphfix_ap",
        "coverage": metrics["coverage"],
        "precision": metrics["precision"],
        "context_size": metrics["context_size"],
        "num_anchors": len(anchor_nodes),
        "hits": metrics["hits"]
    })

    if i % 10 == 0 or i == len(unique_tasks):
        print(
            f"Completed {i}/{len(unique_tasks)}"
        )

Tasks to evaluate: 127
Completed 10/127
Completed 20/127
Completed 30/127
Completed 40/127
Completed 50/127
Completed 60/127
Completed 70/127
Completed 80/127
Completed 90/127
Completed 100/127
Completed 110/127
Completed 120/127
Completed 127/127


In [287]:
graphfix_ap_df = pd.DataFrame(graphfix_ap_rows)

print("Rows:", len(graphfix_ap_df))

print("\nGraphFix-AP summary:")
print(
    graphfix_ap_df[
        ["coverage", "precision", "context_size", "num_anchors"]
    ].mean()
)

Rows: 127

GraphFix-AP summary:
coverage         0.391345
precision        0.089239
context_size    29.763780
num_anchors      4.881890
dtype: float64


In [291]:
# Compare GraphFix-AP against the original GraphFix
comparison = results_df.merge(
    graphfix_ap_df[
        ["instance_id", "coverage", "precision"]
    ],
    on="instance_id",
    suffixes=("_old", "_ap")
)

comparison["coverage_delta"] = (
    comparison["coverage_ap"]
    - comparison["coverage_old"]
)

print("GraphFix-AP vs Original GraphFix")
print(
    "Improved:",
    (comparison["coverage_delta"] > 0).sum()
)

print(
    "Same:",
    (comparison["coverage_delta"] == 0).sum()
)

print(
    "Worse:",
    (comparison["coverage_delta"] < 0).sum()
)

print(
    "Mean coverage change:",
    comparison["coverage_delta"].mean()
)

GraphFix-AP vs Original GraphFix
Improved: 55
Same: 381
Worse: 72
Mean coverage change: -0.020195159171537126


In [299]:
original_graphfix = results_df[
    results_df["method"] == "graphfix"
][
    ["instance_id", "coverage", "precision", "context_size"]
].copy()

original_graphfix = original_graphfix.rename(
    columns={
        "coverage": "coverage_graphfix",
        "precision": "precision_graphfix",
        "context_size": "context_graphfix"
    }
)

graphfix_ap_compare = graphfix_ap_df[
    ["instance_id", "coverage", "precision", "context_size"]
].copy()

graphfix_ap_compare = graphfix_ap_compare.rename(
    columns={
        "coverage": "coverage_ap",
        "precision": "precision_ap",
        "context_size": "context_ap"
    }
)

comparison_clean = original_graphfix.merge(
    graphfix_ap_compare,
    on="instance_id",
    how="inner"
)

comparison_clean["coverage_delta"] = (
    comparison_clean["coverage_ap"]
    - comparison_clean["coverage_graphfix"]
)

print("Rows:", len(comparison_clean))

print("\nCoverage change:")
print(
    comparison_clean["coverage_delta"]
    .value_counts()
    .sort_index()
)

print(
    "\nMean coverage change:",
    comparison_clean["coverage_delta"].mean()
)

Rows: 127

Coverage change:
coverage_delta
-1.000000    13
-0.500000     2
-0.181818     1
 0.000000    97
 0.038462     1
 0.250000     1
 0.500000     2
 1.000000    10
Name: count, dtype: int64

Mean coverage change: -0.02278233577446176


In [303]:
worst_cases = comparison_clean[
    comparison_clean["coverage_delta"] < 0
].sort_values("coverage_delta")

print(
    worst_cases[
        [
            "instance_id",
            "coverage_graphfix",
            "coverage_ap",
            "coverage_delta"
        ]
    ].to_string(index=False)
)

  instance_id  coverage_graphfix  coverage_ap  coverage_delta
    rich_4079           1.000000     0.000000       -1.000000
    rich_4076           1.000000     0.000000       -1.000000
fastapi_11355           1.000000     0.000000       -1.000000
fastapi_14482           1.000000     0.000000       -1.000000
fastapi_13207           1.000000     0.000000       -1.000000
fastapi_14455           1.000000     0.000000       -1.000000
fastapi_14791           1.000000     0.000000       -1.000000
    rich_3935           1.000000     0.000000       -1.000000
    rich_3472           1.000000     0.000000       -1.000000
    rich_3480           1.000000     0.000000       -1.000000
    rich_3777           1.000000     0.000000       -1.000000
    rich_3535           1.000000     0.000000       -1.000000
    rich_3006           1.000000     0.000000       -1.000000
fastapi_14372           0.500000     0.000000       -0.500000
fastapi_14448           0.500000     0.000000       -0.500000
fastapi_

In [304]:
instance_id = "rich_4079"

row = comparison_clean[
    comparison_clean["instance_id"] == instance_id
]

print(row.to_string(index=False))

instance_id  coverage_graphfix  precision_graphfix  context_graphfix  coverage_ap  precision_ap  context_ap  coverage_delta
  rich_4079                1.0            0.066667                30          0.0           0.0          30            -1.0


In [306]:
instance_id = "rich_4079"

task_row = tasks_df[
    tasks_df["instance_id"] == instance_id
].iloc[0]

base_commit = task_row["base_commit"]
repo = task_row["repo"]

print("Instance:", instance_id)
print("Repo:", repo)
print("Base commit:", base_commit)
print("Changed files:", task_row["changed_files"])

Instance: rich_4079
Repo: Textualize/rich
Base commit: 19c67b9a3479841e9133bea94607c89ee931d3fc
Changed files: ['rich/markdown.py']


In [311]:
# Find existing variables that look like graph / embedding resources

for name in sorted(globals()):
    if any(word in name.lower() for word in ["graph", "embed", "cache", "resource"]):
        try:
            value = globals()[name]
            print(f"{name}: {type(value).__name__}")
        except:
            pass

EMBEDDINGS_DIR: PosixPath
EMBED_DIR: str
GRAPHS_DIR: PosixPath
GRAPH_DIR: str
baseline_embedding_retrieval: function
baseline_fixed_graph: function
build_graph_index: function
embedding_commits: set
embedding_files: list
embedding_keys: set
embedding_nodes_selected: list
embedding_retrieve: function
embedding_retrieve_fast: function
embedding_scores: dict
embeddings: NpzFile
fixed_graph_nodes: list
graph: dict
graph_commits: set
graph_data: dict
graph_distance: function
graph_files: list
graph_info: list
graph_info_df: DataFrame
graph_nodes: set
graph_path: str
graph_repo_counts: Counter
graph_stats: list
graph_stats_df: DataFrame
graphfix_anchor_preserved: function
graphfix_ap_compare: DataFrame
graphfix_ap_df: DataFrame
graphfix_ap_rows: list
graphfix_expand: function
graphfix_metrics: dict
graphfix_nodes: list
graphfix_selected: list
load_resources: function
normalized_embedding_scores: function
original_graphfix: DataFrame
prepare_embedding_matrix: function
sample_embedding: NpzFil

In [312]:
print("Variables containing graph/embedding/resource:")
for name in sorted(globals()):
    if any(x in name.lower() for x in ["graph", "embed", "resource"]):
        print(name)

Variables containing graph/embedding/resource:
EMBEDDINGS_DIR
EMBED_DIR
GRAPHS_DIR
GRAPH_DIR
baseline_embedding_retrieval
baseline_fixed_graph
build_graph_index
embedding_commits
embedding_files
embedding_keys
embedding_nodes_selected
embedding_retrieve
embedding_retrieve_fast
embedding_scores
embeddings
fixed_graph_nodes
graph
graph_commits
graph_data
graph_distance
graph_files
graph_info
graph_info_df
graph_nodes
graph_path
graph_repo_counts
graph_stats
graph_stats_df
graphfix_anchor_preserved
graphfix_ap_compare
graphfix_ap_df
graphfix_ap_rows
graphfix_expand
graphfix_metrics
graphfix_nodes
graphfix_selected
load_resources
normalized_embedding_scores
original_graphfix
prepare_embedding_matrix
sample_embedding
sample_embedding_path
sample_graph
sample_graph_path
task_embedding
task_graph


In [313]:
# Reconstruct resources for rich_4079

instance_id = "rich_4079"

task_row = tasks_df[
    tasks_df["instance_id"] == instance_id
].iloc[0]

repo = task_row["repo"]
base_commit = task_row["base_commit"]

graph, embeddings = load_resources(repo, base_commit)

print("Graph loaded:", len(graph["nodes"]), "nodes")
print("Embeddings loaded:", len(embeddings.files), "embedding entries")
print("Changed files:", task_row["changed_files"])


Graph loaded: 2021 nodes
Embeddings loaded: 2021 embedding entries
Changed files: ['rich/markdown.py']


In [314]:
# Reconstruct the exact selections for rich_4079

terms = extract_anchor_terms(task_row["problem_statement"])

anchor_results = find_anchor_nodes_fast(
    graph,
    terms
)

anchor_nodes = [x["node"] for x in anchor_results]

print("Number of anchors:", len(anchor_nodes))
print("\nAnchors:")
for node in anchor_nodes:
    print(node)

embedding_results = embedding_retrieve_fast(
    graph,
    embeddings,
    anchor_nodes,
    top_k=30
)

embedding_scores = {
    x["node"]: x["similarity"]
    for x in embedding_results
}

original_nodes = graphfix_expand(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    max_depth=3
)

ap_nodes = graphfix_anchor_preserved(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    anchor_budget=5,
    max_depth=3
)

print("\nOriginal GraphFix:", len(original_nodes))
for node in original_nodes:
    print(node)

print("\nGraphFix-AP:", len(ap_nodes))
for node in ap_nodes:
    print(node)

Number of anchors: 20

Anchors:
tests.test_ansi.test_decode_newlines
tests.test_cells.test_nerd_font
tests.test_console.test_capture_and_record
tests.test_console.test_recording_no_stdout_and_no_stderr_files
tests.test_console.test_render_lines_height_minus_vertical_pad_is_negative
tests.test_console.test_reset_height
tests.test_file_proxy.test_isatty
tests.test_live.test_live_empty
tests.test_markdown.test_inline_styles_in_table
tests.test_markdown.test_inline_styles_with_justification
tests.test_markdown.test_table_with_empty_cells
tests.test_markup.test_escape
tests.test_markup.test_escape_backslash_end
tests.test_panel.test_title_text_with_border_color
tests.test_panel.test_title_text_with_panel_background
tests.test_pretty.test_dataclass_no_attribute
tests.test_pretty.test_measure_pretty
tests.test_segment.test_split_cells_mixed
tests.test_style.test_clear_meta_and_links_clears_hash
tests.test_syntax.test_background_color_override_includes_padding

Original GraphFix: 30
{'node': '

In [315]:
def is_test_node(node_id):
    return (
        node_id.startswith("tests.")
        or ".tests." in node_id
    )

test_anchor_stats = []

for _, row in tasks_df.drop_duplicates("base_commit").iterrows():
    try:
        graph, _ = load_resources(row["repo"], row["base_commit"])

        terms = extract_anchor_terms(row["problem_statement"])
        anchor_results = find_anchor_nodes_fast(graph, terms)
        anchors = [x["node"] for x in anchor_results]

        test_count = sum(is_test_node(a) for a in anchors)

        test_anchor_stats.append({
            "instance_id": row["instance_id"],
            "num_anchors": len(anchors),
            "test_anchors": test_count,
            "production_anchors": len(anchors) - test_count
        })
    except Exception as e:
        print("Error:", row["instance_id"], e)

test_anchor_df = pd.DataFrame(test_anchor_stats)

print("Tasks evaluated:", len(test_anchor_df))
print(
    "Tasks with ONLY test anchors:",
    (test_anchor_df["production_anchors"] == 0).sum()
)
print(
    "Tasks with at least one production anchor:",
    (test_anchor_df["production_anchors"] > 0).sum()
)

print("\nAverage test anchors:",
      test_anchor_df["test_anchors"].mean())

print("\nAverage production anchors:",
      test_anchor_df["production_anchors"].mean())

Tasks evaluated: 127
Tasks with ONLY test anchors: 9
Tasks with at least one production anchor: 118

Average test anchors: 68.7244094488189

Average production anchors: 72.26771653543307


In [316]:
print(test_anchor_df["num_anchors"].describe())

print("\nAnchor-count distribution:")
print(
    test_anchor_df["num_anchors"]
    .value_counts()
    .sort_index()
    .to_string()
)

count    127.000000
mean     140.992126
std      167.532489
min        0.000000
25%       30.000000
50%       86.000000
75%      189.500000
max      990.000000
Name: num_anchors, dtype: float64

Anchor-count distribution:
num_anchors
0       1
1       1
2       1
3       1
4       1
10      2
14      1
18      1
19      3
20      9
21      1
24      2
27      1
28      1
30     10
32      1
33      1
38      1
39      2
40      7
41      1
50      3
55      1
56      1
60      1
69      1
70      3
74      1
80      1
81      1
85      1
86      1
90      4
91      1
100     1
107     1
108     1
110     2
112     1
117     1
120     4
130     1
138     1
150     1
160     3
161     1
170     4
180     3
189     1
190     1
200     4
210     3
230     2
240     1
246     1
250     1
256     1
270     1
280     1
290     1
300     1
310     2
320     1
350     1
370     1
420     1
450     1
470     1
590     2
630     1
700     1
800     1
990     1


In [317]:
print("results_df columns:")
print(results_df.columns.tolist())

print("\nGraphFix rows:")
print(
    results_df[results_df["method"] == "graphfix"]
    .head()
    .to_string()
)

results_df columns:
['instance_id', 'repo', 'base_commit', 'method', 'coverage', 'precision', 'context_size', 'hits', 'num_anchors', 'num_anchor_candidates']

GraphFix rows:
      instance_id             repo                               base_commit    method  coverage  precision  context_size  hits  num_anchors  num_anchor_candidates
3   fastapi_15661  fastapi/fastapi  ee22a4b8ca46dcce26c8c183afc4992a888d8be2  graphfix  0.000000   0.000000            30     0            5                    512
7   fastapi_15588  fastapi/fastapi  cb83b83dcf78eab4ea17d504db5abcda705fbdc4  graphfix  0.000000   0.000000            30     0            5                    721
11  fastapi_15589  fastapi/fastapi  22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b  graphfix  1.000000   0.166667            30     1            5                    524
15  fastapi_15030  fastapi/fastapi  48d58ae3b6ef4ded5a4794618c00345667256d98  graphfix  0.000000   0.000000            30     0            5                    540
19  fa

In [320]:
# Inspect the stored rich_4079 result and its exact task metadata

print("Stored GraphFix result:")
print(
    results_df[
        (results_df["instance_id"] == "rich_4079") &
        (results_df["method"] == "graphfix")
    ].to_string(index=False)
)

print("\nTask metadata:")
print(
    tasks_df[
        tasks_df["instance_id"] == "rich_4079"
    ][["instance_id", "repo", "base_commit", "changed_files"]]
    .to_string(index=False)
)

Stored GraphFix result:
instance_id            repo                              base_commit   method  coverage  precision  context_size  hits  num_anchors  num_anchor_candidates
  rich_4079 Textualize/rich 19c67b9a3479841e9133bea94607c89ee931d3fc graphfix       1.0   0.066667            30     1            5                     36

Task metadata:
instance_id            repo                              base_commit      changed_files
  rich_4079 Textualize/rich 19c67b9a3479841e9133bea94607c89ee931d3fc [rich/markdown.py]


In [322]:
# Inspect the exact anchor candidates for rich_4079

row = tasks_df[tasks_df["instance_id"] == "rich_4079"].iloc[0]

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

terms = extract_anchor_terms(row["problem_statement"])
anchor_results = find_anchor_nodes_fast(
    graph,
    terms
)

anchors = [x["node"] for x in anchor_results]

print("Number of anchor candidates:", len(anchor_results))
print("\nSelected 5 anchors:")
for x in anchor_results[:5]:
    print(x["node"], "| score:", x["score"], "| matched:", x["matched_terms"])

Number of anchor candidates: 20

Selected 5 anchors:
tests.test_ansi.test_decode_newlines | score: 2 | matched: ['github.com', 'Textualize']
tests.test_cells.test_nerd_font | score: 2 | matched: ['github.com', 'Textualize']
tests.test_console.test_capture_and_record | score: 2 | matched: ['github.com', 'Textualize']
tests.test_console.test_recording_no_stdout_and_no_stderr_files | score: 2 | matched: ['github.com', 'Textualize']
tests.test_console.test_render_lines_height_minus_vertical_pad_is_negative | score: 2 | matched: ['github.com', 'Textualize']


In [323]:
# Reproduce the original GraphFix selection for rich_4079

embedding_results = embedding_retrieve_fast(
    graph,
    embeddings,
    anchors,
    top_k=30
)

embedding_scores = {
    x["node"]: x["similarity"]
    for x in embedding_results
}

graphfix_selected = graphfix_expand(
    graph,
    anchors,
    embedding_scores,
    budget=30,
    max_depth=3
)

print("GraphFix selected:", len(graphfix_selected))

print("\nSelected nodes:")
for i, item in enumerate(graphfix_selected, 1):
    node = item["node"] if isinstance(item, dict) else item
    print(i, node)

print("\nChanged-file matches:")
for item in graphfix_selected:
    node = item["node"] if isinstance(item, dict) else item
    if node_matches_file(node, "rich/markdown.py"):
        print("MATCH:", node)

GraphFix selected: 30

Selected nodes:
1 tests.test_console.test_reset_height
2 tests.test_panel.test_title_text_with_border_color
3 tests.test_panel.test_title_text_with_panel_background
4 tests.test_console.test_recording_no_stdout_and_no_stderr_files
5 tests.test_cells.test_nerd_font
6 tests.test_style.test_clear_meta_and_links_clears_hash
7 tests.test_pretty.test_dataclass_no_attribute
8 tests.test_segment.test_split_cells_mixed
9 tests.test_console.test_capture_and_record
10 tests.test_ansi.test_decode_newlines
11 tests.test_markdown.test_inline_styles_with_justification
12 tests.test_markdown.test_inline_styles_in_table
13 tests.test_file_proxy.test_isatty
14 tests.test_syntax.test_background_color_override_includes_padding
15 tests.test_pretty.test_measure_pretty
16 tests.test_live.test_live_empty
17 tests.test_markup.test_escape_backslash_end
18 tests.test_markup.test_escape
19 tests.test_markdown.test_table_with_empty_cells
20 tests.test_console.test_render_lines_height_minus_

In [326]:
print([name for name in globals() if "result" in name.lower() or "eval" in name.lower()])

['results', 'anchor_results', 'retrieval', 'results_df', 'baseline_anchor_retrieval', 'baseline_embedding_retrieval', 'embedding_results']


In [327]:
print(type(results))

if isinstance(results, list):
    print("Number of result items:", len(results))
    print("\nFirst result:")
    print(results[0])
elif isinstance(results, dict):
    print("Result keys:")
    print(results.keys())
    print("\nSample:")
    print(next(iter(results.items())))
else:
    print(results)

<class 'list'>
Number of result items: 508

First result:
{'instance_id': 'fastapi_15661', 'repo': 'fastapi/fastapi', 'base_commit': 'ee22a4b8ca46dcce26c8c183afc4992a888d8be2', 'method': 'anchor', 'coverage': 0.0, 'precision': 0.0, 'context_size': 30, 'hits': 0, 'num_anchors': 5, 'num_anchor_candidates': 512}


In [328]:
import inspect

print("graphfix_expand:")
print(inspect.getsource(graphfix_expand))

print("\nlocalization_metrics:")
print(inspect.getsource(localization_metrics))

graphfix_expand:
def graphfix_expand(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    max_depth=3,
    alpha=0.55,
    beta=0.30,
    gamma=0.15
):
    """
    Adaptive graph-guided context selection.

    Score components:

    - semantic_score:
        supplied embedding similarity

    - structural_score:
        proximity to discovered anchors

    - diversity_score:
        preference for nodes not already represented
        by the selected neighborhood

    The selector greedily chooses high-value nodes until
    the context budget is reached.
    """

    nodes_by_id, outgoing, incoming = build_graph_index(graph)

    # --------------------------------------------------------
    # Graph distances from all anchors
    # --------------------------------------------------------

    distances = graph_distance(
        anchor_nodes,
        outgoing,
        incoming,
        max_depth=max_depth
    )

    # ---------------------------------------------------

In [339]:
row = tasks_df[tasks_df["instance_id"] == "rich_4079"].iloc[0]

print("INSTANCE:", row["instance_id"])
print("BASE COMMIT:", row["base_commit"])
print("CHANGED FILES:", row["changed_files"])

terms = extract_anchor_terms(row["problem_statement"])

print("\nANCHOR TERMS:")
for i, term in enumerate(terms):
    print(i, repr(term))

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

anchor_candidates = find_anchor_nodes_fast(
    graph,
    terms
)

print("\nCURRENT ANCHOR CANDIDATES:", len(anchor_candidates))

for i, item in enumerate(anchor_candidates[:30]):
    print(
        i,
        item["node"],
        "score=", item["score"],
        "matched=", item["matched_terms"]
    )

print("\nSTORED GRAPHFIX RESULT:")
stored = results_df[
    (results_df["instance_id"] == "rich_4079") &
    (results_df["method"] == "graphfix")
].iloc[0]

print(stored.to_dict())

INSTANCE: rich_4079
BASE COMMIT: 19c67b9a3479841e9133bea94607c89ee931d3fc
CHANGED FILES: ['rich/markdown.py']

ANCHOR TERMS:
0 'github.com'
1 'Textualize'

CURRENT ANCHOR CANDIDATES: 20
0 tests.test_ansi.test_decode_newlines score= 2 matched= ['github.com', 'Textualize']
1 tests.test_cells.test_nerd_font score= 2 matched= ['github.com', 'Textualize']
2 tests.test_console.test_capture_and_record score= 2 matched= ['github.com', 'Textualize']
3 tests.test_console.test_recording_no_stdout_and_no_stderr_files score= 2 matched= ['github.com', 'Textualize']
4 tests.test_console.test_render_lines_height_minus_vertical_pad_is_negative score= 2 matched= ['github.com', 'Textualize']
5 tests.test_console.test_reset_height score= 2 matched= ['github.com', 'Textualize']
6 tests.test_file_proxy.test_isatty score= 2 matched= ['github.com', 'Textualize']
7 tests.test_live.test_live_empty score= 2 matched= ['github.com', 'Textualize']
8 tests.test_markdown.test_inline_styles_in_table score= 2 matched= 

In [341]:
import inspect

print("extract_anchor_terms:")
print(inspect.getsource(extract_anchor_terms))

print("\nfind_anchor_nodes_fast:")
print(inspect.getsource(find_anchor_nodes_fast))

extract_anchor_terms:
def extract_anchor_terms(text):
    """
    Extract code-like terms from an issue description.

    We prefer:
    - snake_case identifiers
    - dotted identifiers
    - CamelCase names
    - ALL_CAPS names
    - longer distinctive tokens
    """

    if not isinstance(text, str):
        return []

    patterns = [
        r'\b[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_.]*\b',
        r'\b[A-Za-z_][A-Za-z0-9_]*_[A-Za-z0-9_]+\b',
        r'\b[A-Z][A-Za-z0-9]+(?:[A-Z][A-Za-z0-9]+)+\b',
        r'\b[A-Z]{2,}[A-Za-z0-9_]*\b',
    ]

    terms = []

    for pattern in patterns:
        terms.extend(re.findall(pattern, text))

    # Also keep unusually long normal words.
    words = re.findall(r'\b[A-Za-z][A-Za-z0-9_-]{7,}\b', text)

    terms.extend(words)

    cleaned = []

    for term in terms:
        term = term.strip(".,:;()[]{}<>\"'`")

        if len(term) < 5:
            continue

        if term.lower() in STOPWORDS:
            continue

        cleaned.

In [1]:
print("extract_anchor_terms" in globals())
print("find_anchor_nodes_fast" in globals())
print("graphfix_expand" in globals())
print("localization_metrics" in globals())

False
False
False
False


In [6]:
 print("BASE" in globals())
print("tasks_df" in globals())
print("load_resources" in globals())
    

False
False
False


In [23]:
import os

BASE = "/kaggle/input"

COMP_DIR = os.path.join(
    BASE,
    "competitions",
    "gemma-4-developer-agent"
)

GRAPH_DIR = os.path.join(COMP_DIR, "graphs")
EMBEDDING_DIR = os.path.join(COMP_DIR, "embeddings")
SNAPSHOT_DIR = os.path.join(COMP_DIR, "snapshots")

print("GRAPH_DIR:", GRAPH_DIR)
print("Graph files:", len(os.listdir(GRAPH_DIR)))

print("EMBEDDING_DIR:", EMBEDDING_DIR)
print("Embedding files:", len(os.listdir(EMBEDDING_DIR)))

print("SNAPSHOT_DIR:", SNAPSHOT_DIR)
print("Snapshot files:", len(os.listdir(SNAPSHOT_DIR)))

GRAPH_DIR: /kaggle/input/competitions/gemma-4-developer-agent/graphs
Graph files: 127
EMBEDDING_DIR: /kaggle/input/competitions/gemma-4-developer-agent/embeddings
Embedding files: 127
SNAPSHOT_DIR: /kaggle/input/competitions/gemma-4-developer-agent/snapshots
Snapshot files: 129


In [25]:
EMBED_DIR = os.path.join(
    COMP_DIR,
    "embeddings"
)

print("EMBED_DIR:", EMBED_DIR)
print("Embedding files:", len(os.listdir(EMBED_DIR)))

EMBED_DIR: /kaggle/input/competitions/gemma-4-developer-agent/embeddings
Embedding files: 127


In [26]:
sample_task = tasks_df.iloc[0]

graph, embeddings = load_resources(
    sample_task["repo"],
    sample_task["base_commit"]
)

print("Repo:", sample_task["repo"])
print("Base commit:", sample_task["base_commit"])
print("Graph nodes:", len(graph["nodes"]))
print("Embedding vectors:", len(embeddings))

Repo: fastapi/fastapi
Base commit: ee22a4b8ca46dcce26c8c183afc4992a888d8be2
Graph nodes: 4263
Embedding vectors: 4263


In [62]:
from collections import defaultdict, deque

def build_graph_index(graph):
    nodes_by_id = {}
    outgoing = defaultdict(list)
    incoming = defaultdict(list)

    for node in graph["nodes"]:
        node_id = str(node["id"])
        nodes_by_id[node_id] = node

    for edge in graph["edges"]:
        source = str(edge["source"])
        target = str(edge["target"])

        outgoing[source].append(target)
        incoming[target].append(source)

    return nodes_by_id, outgoing, incoming


def graph_distance(start_nodes, outgoing, incoming, max_depth=2):
    distances = {}
    queue = deque()

    for node in start_nodes:
        if node not in distances:
            distances[node] = 0
            queue.append(node)

    while queue:
        current = queue.popleft()
        distance = distances[current]

        if distance >= max_depth:
            continue

        neighbors = set(
            outgoing.get(current, [])
            + incoming.get(current, [])
        )

        for neighbor in neighbors:
            if neighbor not in distances:
                distances[neighbor] = distance + 1
                queue.append(neighbor)

    return distances


def node_file_path(node_id):
    parts = str(node_id).split(".")

    if len(parts) >= 2:
        return "/".join(parts[:2])

    return str(node_id)

In [63]:
nodes_by_id, outgoing, incoming = build_graph_index(graph)

print("Indexed nodes:", len(nodes_by_id))
print("Outgoing links:", sum(len(v) for v in outgoing.values()))
print("Incoming links:", sum(len(v) for v in incoming.values()))

Indexed nodes: 4263
Outgoing links: 2430
Incoming links: 2430


In [78]:
def find_anchor_nodes_fast(graph, terms, max_per_term=10):
    terms = [str(t).strip() for t in terms if str(t).strip()]
    normalized_terms = [term.lower() for term in terms]

    results = {}

    for node in graph["nodes"]:
        node_id = str(node.get("id", ""))
        name = str(node.get("name", ""))
        source = str(node.get("text", ""))

        node_id_lower = node_id.lower()
        name_lower = name.lower()

        matched = []
        score = 0

        for term, term_lower in zip(terms, normalized_terms):
            if (
                node_id_lower == term_lower
                or name_lower == term_lower
                or node_id_lower.endswith("." + term_lower)
                or name_lower == term_lower
            ):
                matched.append(term)
                score += 3
                continue

            if term_lower in source.lower():
                matched.append(term)
                score += 1

        if matched:
            results[node_id] = {
                "node": node_id,
                "score": score,
                "matched_terms": matched
            }

    ranked = sorted(
        results.values(),
        key=lambda x: (-x["score"], x["node"])
    )

    return ranked[:max_per_term * max(1, len(terms))]


In [93]:
def prepare_embedding_matrix(graph, embeddings):
    node_ids = []
    vectors = []

    for node in graph["nodes"]:
        node_id = str(node["id"])

        if node_id in embeddings:
            vector = np.asarray(
                embeddings[node_id],
                dtype=np.float32
            )

            if vector.ndim == 1:
                node_ids.append(node_id)
                vectors.append(vector)

    matrix = np.vstack(vectors)

    return node_ids, matrix


def embedding_retrieve_fast(
    graph,
    embeddings,
    anchor_nodes,
    top_k=30
):
    node_ids, matrix = prepare_embedding_matrix(
        graph,
        embeddings
    )

    node_to_index = {
        node_id: i
        for i, node_id in enumerate(node_ids)
    }

    matrix_norms = np.linalg.norm(
        matrix,
        axis=1,
        keepdims=True
    )

    matrix_norms[matrix_norms == 0] = 1.0

    matrix_normalized = (
        matrix / matrix_norms
    )

    best_scores = {}
    best_anchor = {}

    for anchor in anchor_nodes:
        if anchor not in node_to_index:
            continue

        query = matrix[node_to_index[anchor]]

        query_norm = np.linalg.norm(query)

        if query_norm == 0:
            continue

        query = query / query_norm

        similarities = matrix_normalized @ query

        top_indices = np.argsort(
            similarities
        )[-top_k:][::-1]

        for idx in top_indices:
            node_id = node_ids[idx]
            score = float(similarities[idx])

            if (
                node_id not in best_scores
                or score > best_scores[node_id]
            ):
                best_scores[node_id] = score
                best_anchor[node_id] = anchor

    results = []

    for node_id, score in best_scores.items():
        results.append({
            "node": node_id,
            "similarity": score,
            "anchor": best_anchor[node_id]
        })

    results.sort(
        key=lambda x: x["similarity"],
        reverse=True
    )

    return results[:top_k]

In [105]:
def graphfix_expand(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    max_depth=3,
    alpha=0.55,
    beta=0.30,
    gamma=0.15
):
    nodes_by_id, outgoing, incoming = build_graph_index(graph)

    distances = graph_distance(
        anchor_nodes,
        outgoing,
        incoming,
        max_depth=max_depth
    )

    candidates = set(embedding_scores.keys())

    frontier = set(anchor_nodes)

    for _ in range(max_depth):
        next_frontier = set()

        for node in frontier:
            next_frontier.update(
                outgoing.get(node, [])
            )
            next_frontier.update(
                incoming.get(node, [])
            )

        candidates.update(next_frontier)
        frontier = next_frontier

    candidates = {
        node for node in candidates
        if node in nodes_by_id
    }

    scored = []

    for node in candidates:
        semantic_score = embedding_scores.get(
            node,
            0.0
        )

        distance = distances.get(
            node,
            max_depth + 1
        )

        structural_score = (
            1 / (1 + distance)
            if distance <= max_depth
            else 0.0
        )

        diversity_score = 1.0

        total_score = (
            alpha * semantic_score
            + beta * structural_score
            + gamma * diversity_score
        )

        scored.append({
            "node": node,
            "semantic_score": semantic_score,
            "structural_score": structural_score,
            "diversity_score": diversity_score,
            "score": total_score
        })

    scored.sort(
        key=lambda x: (
            -x["score"],
            x["node"]
        )
    )

    selected = []
    selected_modules = set()

    for item in scored:
        node = item["node"]

        parts = node.split(".")

        module = (
            ".".join(parts[:2])
            if len(parts) >= 2
            else node
        )

        repetition_penalty = (
            0.85
            if module in selected_modules
            else 1.0
        )

        adjusted_score = (
            item["score"] * repetition_penalty
        )

        item = dict(item)
        item["adjusted_score"] = adjusted_score

        selected.append(item)
        selected_modules.add(module)

        if len(selected) >= budget:
            break

    return selected

In [106]:
# Reproduction check: rich_4079

task = tasks_df[
    tasks_df["instance_id"] == "rich_4079"
].iloc[0]

graph, embeddings = load_resources(
    task["repo"],
    task["base_commit"]
)

terms = extract_anchor_terms(
    task["problem_statement"]
)

anchor_candidates = find_anchor_nodes_fast(
    graph,
    terms,
    max_per_term=10
)

anchor_nodes = [
    item["node"]
    for item in anchor_candidates[:5]
]

embedding_results = embedding_retrieve_fast(
    graph,
    embeddings,
    anchor_nodes,
    top_k=30
)

embedding_scores = {
    item["node"]: item["similarity"]
    for item in embedding_results
}

selected = graphfix_expand(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    max_depth=3
)

selected_nodes = [
    item["node"]
    for item in selected
]

print("INSTANCE:", task["instance_id"])
print("CHANGED FILES:", task["changed_files"])
print()
print("ANCHOR TERMS:", terms)
print()
print("ANCHOR CANDIDATES:", len(anchor_candidates))
print("SELECTED ANCHORS:", anchor_nodes)
print()
print("EMBEDDING RESULTS:", len(embedding_results))
print("GRAPHFIX SELECTED:", len(selected_nodes))
print()
print("SELECTED NODES:")
for node in selected_nodes:
    print(" ", node)

INSTANCE: rich_4079


KeyError: 'changed_files'

In [115]:
import re

def extract_changed_files(patch):
    if not isinstance(patch, str):
        return []

    files = []

    for match in re.finditer(
        r"^\+\+\+ b/(.+)$",
        patch,
        re.MULTILINE
    ):
        files.append(match.group(1))

    return files


tasks_df["changed_files"] = tasks_df["patch"].apply(
    extract_changed_files
)

print("changed_files restored")
print("Tasks:", len(tasks_df))
print("Example:", tasks_df.iloc[0]["changed_files"])

changed_files restored
Tasks: 129
Example: ['scripts/prepare_release.py']


In [116]:
task = tasks_df[
    tasks_df["instance_id"] == "rich_4079"
].iloc[0]

graph, embeddings = load_resources(
    task["repo"],
    task["base_commit"]
)

terms = extract_anchor_terms(
    task["problem_statement"]
)

anchor_candidates = find_anchor_nodes_fast(
    graph,
    terms,
    max_per_term=10
)

anchor_nodes = [
    item["node"]
    for item in anchor_candidates[:5]
]

print("INSTANCE:", task["instance_id"])
print("CHANGED FILES:", task["changed_files"])
print()
print("ANCHOR TERMS:")
print(terms)
print()
print("ANCHOR CANDIDATES:", len(anchor_candidates))
print("FIRST 10 CANDIDATES:")

for item in anchor_candidates[:10]:
    print(item)

print()
print("SELECTED 5 ANCHORS:")
for node in anchor_nodes:
    print(node)

INSTANCE: rich_4079
CHANGED FILES: ['rich/markdown.py']

ANCHOR TERMS:
['github.com', 'Textualize']

ANCHOR CANDIDATES: 20
FIRST 10 CANDIDATES:
{'node': 'tests.test_ansi.test_decode_newlines', 'score': 2, 'matched_terms': ['github.com', 'Textualize']}
{'node': 'tests.test_cells.test_nerd_font', 'score': 2, 'matched_terms': ['github.com', 'Textualize']}
{'node': 'tests.test_console.test_capture_and_record', 'score': 2, 'matched_terms': ['github.com', 'Textualize']}
{'node': 'tests.test_console.test_recording_no_stdout_and_no_stderr_files', 'score': 2, 'matched_terms': ['github.com', 'Textualize']}
{'node': 'tests.test_console.test_render_lines_height_minus_vertical_pad_is_negative', 'score': 2, 'matched_terms': ['github.com', 'Textualize']}
{'node': 'tests.test_console.test_reset_height', 'score': 2, 'matched_terms': ['github.com', 'Textualize']}
{'node': 'tests.test_file_proxy.test_isatty', 'score': 2, 'matched_terms': ['github.com', 'Textualize']}
{'node': 'tests.test_live.test_live_e

In [117]:
print(tasks_df[
    tasks_df["instance_id"] == "rich_4079"
].iloc[0]["problem_statement"])

Inline table code

Fixes https://github.com/Textualize/rich/issues/4038


In [121]:
# Inspect the GitHub issue URL and problem text for anchor-like identifiers

text = tasks_df[
    tasks_df["instance_id"] == "rich_4079"
].iloc[0]["problem_statement"]

print("PROBLEM STATEMENT:")
print(text)

print("\nTOKENS THAT LOOK LIKE CODE IDENTIFIERS:")
tokens = re.findall(r'\b[A-Za-z_][A-Za-z0-9_.]*\b', text)

for token in tokens:
    if (
        "." in token
        or "_" in token
        or token[:1].isupper()
    ):
        print(token)

PROBLEM STATEMENT:
Inline table code

Fixes https://github.com/Textualize/rich/issues/4038

TOKENS THAT LOOK LIKE CODE IDENTIFIERS:
Inline
Fixes
github.com
Textualize


In [125]:
row = tasks_df[
    tasks_df["instance_id"] == "rich_4079"
].iloc[0]

print("HINTS:")
print(row["hints_text"])

HINTS:



In [130]:
row = tasks_df[
    tasks_df["instance_id"] == "rich_4079"
].iloc[0]

print("PATCH:")
print(row["patch"])

PATCH:
--- a/rich/markdown.py
+++ b/rich/markdown.py
@@ -331,9 +331,10 @@ def __init__(self, justify: JustifyMethod) -> None:
         self.justify = justify
 
     def on_text(self, context: MarkdownContext, text: TextType) -> None:
-        text = Text(text) if isinstance(text, str) else text
-        text.stylize(context.current_style)
-        self.content.append_text(text)
+        if isinstance(text, str):
+            self.content.append(text, context.current_style)
+        else:
+            self.content.append_text(text)
 
 
 class ListElement(MarkdownElement):


In [134]:
stored_4079 = results_df[
    (results_df["instance_id"] == "rich_4079") &
    (results_df["method"] == "graphfix")
]

print(stored_4079.to_string(index=False))

NameError: name 'results_df' is not defined

In [137]:
import inspect

print(inspect.getsource(extract_anchor_terms))
print("\n--- find_anchor_nodes_fast ---")
print(inspect.getsource(find_anchor_nodes_fast))

def extract_anchor_terms(text):
    """
    Extract code-like terms from an issue description.

    We prefer:
    - snake_case identifiers
    - dotted identifiers
    - CamelCase names
    - ALL_CAPS names
    - longer distinctive tokens
    """

    if not isinstance(text, str):
        return []

    patterns = [
        r'\b[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_.]*\b',
        r'\b[A-Za-z_][A-Za-z0-9_]*_[A-Za-z0-9_]+\b',
        r'\b[A-Z][A-Za-z0-9]+(?:[A-Z][A-Za-z0-9]+)+\b',
        r'\b[A-Z]{2,}[A-Za-z0-9_]*\b',
    ]

    terms = []

    for pattern in patterns:
        terms.extend(re.findall(pattern, text))

    # Also keep unusually long normal words.
    words = re.findall(r'\b[A-Za-z][A-Za-z0-9_-]{7,}\b', text)

    terms.extend(words)

    cleaned = []

    for term in terms:
        term = term.strip(".,:;()[]{}<>\"'`")

        if len(term) < 5:
            continue

        if term.lower() in STOPWORDS:
            continue

        cleaned.append(term)

    # Pr

In [143]:
row = tasks_df[tasks_df["instance_id"] == "rich_4079"].iloc[0]

graph, embeddings = load_resources(
    row["repo"],
    row["base_commit"]
)

test_terms = ["Inline", "table"]

candidates = find_anchor_nodes_fast(
    graph,
    test_terms,
    max_per_term=10
)

print("TEST TERMS:", test_terms)
print("CANDIDATES:", len(candidates))

for item in candidates[:20]:
    print(item)

TEST TERMS: ['Inline', 'table']
CANDIDATES: 20
{'node': 'rich.table.Table', 'score': 3, 'matched_terms': ['table']}
{'node': 'rich.markdown.Markdown.__init__', 'score': 2, 'matched_terms': ['Inline', 'table']}
{'node': 'tests.test_markdown.test_inline_styles_in_table', 'score': 2, 'matched_terms': ['Inline', 'table']}
{'node': 'benchmarks.benchmarks.TableSuite', 'score': 1, 'matched_terms': ['table']}
{'node': 'benchmarks.benchmarks.TableSuite._print_table', 'score': 1, 'matched_terms': ['table']}
{'node': 'benchmarks.benchmarks.TableSuite.time_table_heavy_wrapping', 'score': 1, 'matched_terms': ['table']}
{'node': 'benchmarks.benchmarks.TableSuite.time_table_no_wrapping', 'score': 1, 'matched_terms': ['table']}
{'node': 'examples.export.print_table', 'score': 1, 'matched_terms': ['table']}
{'node': 'examples.fullscreen.Header.__rich__', 'score': 1, 'matched_terms': ['table']}
{'node': 'examples.fullscreen.make_sponsor_message', 'score': 1, 'matched_terms': ['table']}
{'node': 'example

In [145]:
anchor_candidates = find_anchor_nodes_fast(
    graph,
    ["Inline", "table"],
    max_per_term=10
)

anchor_nodes = [
    item["node"]
    for item in anchor_candidates[:5]
]

print("SELECTED ANCHORS:")
for node in anchor_nodes:
    print(node)

embedding_results = embedding_retrieve_fast(
    graph,
    embeddings,
    anchor_nodes,
    top_k=30
)

embedding_scores = {
    item["node"]: item["similarity"]
    for item in embedding_results
}

selected = graphfix_expand(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    max_depth=3
)

selected_nodes = [item["node"] for item in selected]

print("\nSELECTED GRAPHFIX NODES:")
for node in selected_nodes[:30]:
    print(node)

SELECTED ANCHORS:
rich.table.Table
rich.markdown.Markdown.__init__
tests.test_markdown.test_inline_styles_in_table
benchmarks.benchmarks.TableSuite
benchmarks.benchmarks.TableSuite._print_table

SELECTED GRAPHFIX NODES:
rich.table.Table
tests.test_markdown.test_inline_styles_in_table
rich.markdown.Markdown.__init__
benchmarks.benchmarks.TableSuite
benchmarks.benchmarks.TableSuite._print_table
tests.test_table.test_get_row_style
tests.test_markdown.test_markdown_table
tests.test_markdown.test_inline_styles_with_justification
tests.test_markdown.test_table_with_empty_cells
tests.test_markdown.test_markdown_render
tests.test_markdown.test_partial_table
tests.test_markdown.test_inline_code
rich.markdown.Markdown._flatten_tokens
tests.test_markdown.replace_link_ids
tests.test_inspect.render
tests.test_text.test_end
tests.test_containers.test_lines_rich_console
tests.test_markdown_no_hyperlinks.render
rich.progress.DownloadColumn.render
rich.console.NewLine
tests.test_style.test_pick_first
r

In [146]:
target = "rich.markdown.MarkdownElement.on_text"

nodes_by_id, outgoing, incoming = build_graph_index(graph)

distances = graph_distance(
    anchor_nodes,
    outgoing,
    incoming,
    max_depth=5
)

print("TARGET:", target)
print("TARGET EXISTS:", target in nodes_by_id)
print("DISTANCE FROM SELECTED ANCHORS:", distances.get(target))

if target in distances:
    print("Target is reachable within depth 5.")
else:
    print("Target is NOT reachable within depth 5.")

TARGET: rich.markdown.MarkdownElement.on_text
TARGET EXISTS: True
DISTANCE FROM SELECTED ANCHORS: 4
Target is reachable within depth 5.


In [147]:
target = "rich.markdown.MarkdownElement.on_text"

print("TARGET:", target)
print("TARGET SELECTED:", target in selected_nodes)

if target in selected_nodes:
    position = selected_nodes.index(target) + 1
    print("TARGET POSITION:", position)
else:
    print("Target was not selected in the top 30 nodes.")

TARGET: rich.markdown.MarkdownElement.on_text
TARGET SELECTED: False
Target was not selected in the top 30 nodes.


In [153]:
target = "rich.markdown.MarkdownElement.on_text"

target_item = next(
    (item for item in selected if item["node"] == target),
    None
)

if target_item:
    print("TARGET SCORE DETAILS:")
    for key, value in target_item.items():
        print(f"{key}: {value}")
else:
    # Reconstruct the candidate scoring used by graphfix_expand
    nodes_by_id, outgoing, incoming = build_graph_index(graph)

    distances = graph_distance(
        anchor_nodes,
        outgoing,
        incoming,
        max_depth=3
    )

    candidates = set(embedding_scores.keys())
    frontier = set(anchor_nodes)

    for _ in range(3):
        next_frontier = set()
        for node in frontier:
            next_frontier.update(outgoing.get(node, []))
            next_frontier.update(incoming.get(node, []))
        candidates.update(next_frontier)
        frontier = next_frontier

    distance = distances.get(target, 4)
    semantic_score = embedding_scores.get(target, 0.0)
    structural_score = 1 / (1 + distance) if distance <= 3 else 0.0
    total_score = (
        0.55 * semantic_score
        + 0.30 * structural_score
        + 0.15
    )

    print("TARGET IS NOT IN THE DEPTH-3 CANDIDATE SET.")
    print("Distance:", distance)
    print("Embedding similarity:", semantic_score)
    print("Structural score:", structural_score)
    print("Total score:", total_score)

TARGET IS NOT IN THE DEPTH-3 CANDIDATE SET.
Distance: 4
Embedding similarity: 0.0
Structural score: 0.0
Total score: 0.15


In [156]:
selected_depth4 = graphfix_expand(
    graph,
    anchor_nodes,
    embedding_scores,
    budget=30,
    max_depth=4
)

selected_nodes_depth4 = [
    item["node"]
    for item in selected_depth4
]

target = "rich.markdown.MarkdownElement.on_text"

print("TARGET SELECTED:", target in selected_nodes_depth4)

if target in selected_nodes_depth4:
    print(
        "TARGET POSITION:",
        selected_nodes_depth4.index(target) + 1
    )

print("\nFIRST 30 NODES:")
for i, node in enumerate(selected_nodes_depth4, 1):
    print(i, node)

TARGET SELECTED: False

FIRST 30 NODES:
1 rich.table.Table
2 tests.test_markdown.test_inline_styles_in_table
3 rich.markdown.Markdown.__init__
4 benchmarks.benchmarks.TableSuite
5 benchmarks.benchmarks.TableSuite._print_table
6 tests.test_table.test_get_row_style
7 tests.test_markdown.test_markdown_table
8 tests.test_markdown.test_inline_styles_with_justification
9 tests.test_markdown.test_table_with_empty_cells
10 tests.test_markdown.test_markdown_render
11 tests.test_markdown.test_partial_table
12 tests.test_markdown.test_inline_code
13 rich.markdown.Markdown._flatten_tokens
14 tests.test_markdown.replace_link_ids
15 tests.test_inspect.render
16 tests.test_text.test_end
17 tests.test_containers.test_lines_rich_console
18 tests.test_markdown_no_hyperlinks.render
19 rich.progress.DownloadColumn.render
20 rich.console.NewLine
21 tests.test_style.test_pick_first
22 rich.layout.Layout.__rich_console__
23 tests.test_syntax.test_ansi_theme
24 rich.ansi._ansi_tokenize
25 rich.ansi._AnsiToken

In [160]:
target = "rich.markdown.MarkdownElement.on_text"

nodes_by_id, outgoing, incoming = build_graph_index(graph)

distances = graph_distance(
    anchor_nodes,
    outgoing,
    incoming,
    max_depth=4
)

# Recreate the candidate set at depth 4
candidates = set(embedding_scores.keys())
frontier = set(anchor_nodes)

for _ in range(4):
    next_frontier = set()
    for node in frontier:
        next_frontier.update(outgoing.get(node, []))
        next_frontier.update(incoming.get(node, []))
    candidates.update(next_frontier)
    frontier = next_frontier

scored = []

for node in candidates:
    distance = distances.get(node, 5)
    semantic_score = embedding_scores.get(node, 0.0)
    structural_score = 1 / (1 + distance) if distance <= 4 else 0.0

    total_score = (
        0.55 * semantic_score
        + 0.30 * structural_score
        + 0.15
    )

    scored.append((total_score, node, distance, semantic_score))

scored.sort(reverse=True)

target_rank = next(
    (i + 1 for i, item in enumerate(scored) if item[1] == target),
    None
)

print("TARGET:", target)
print("TARGET RANK:", target_rank)

if target_rank:
    target_item = scored[target_rank - 1]
    print("TARGET SCORE:", target_item[0])
    print("TARGET DISTANCE:", target_item[2])
    print("TARGET SEMANTIC SCORE:", target_item[3])

    print("\nNODES AROUND TARGET:")
    start = max(0, target_rank - 6)
    end = min(len(scored), target_rank + 5)

    for i in range(start, end):
        score, node, distance, semantic = scored[i]
        print(
            f"{i+1:3d} | score={score:.4f} | "
            f"dist={distance} | semantic={semantic:.4f} | {node}"
        )

TARGET: rich.markdown.MarkdownElement.on_text
TARGET RANK: 1455
TARGET SCORE: 0.21
TARGET DISTANCE: 4
TARGET SEMANTIC SCORE: 0.0

NODES AROUND TARGET:
1450 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.TextElement
1451 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.TableRowElement
1452 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.TableHeaderElement
1453 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.TableDataElement
1454 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.TableBodyElement
1455 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.MarkdownElement.on_text
1456 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.MarkdownElement.on_leave
1457 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.MarkdownElement.on_enter
1458 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.MarkdownElement.on_child_close
1459 | score=0.2100 | dist=4 | semantic=0.0000 | rich.markdown.MarkdownElement.create
1460 | sco

In [161]:
anchor = "rich.markdown.Markdown.__init__"
target = "rich.markdown.MarkdownElement.on_text"

node_ids, matrix = prepare_embedding_matrix(graph, embeddings)

node_to_index = {
    node_id: i for i, node_id in enumerate(node_ids)
}

def cosine_similarity(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)

    a_norm = np.linalg.norm(a)
    b_norm = np.linalg.norm(b)

    if a_norm == 0 or b_norm == 0:
        return 0.0

    return float(np.dot(a, b) / (a_norm * b_norm))

anchor_vec = matrix[node_to_index[anchor]]
target_vec = matrix[node_to_index[target]]

similarity = cosine_similarity(anchor_vec, target_vec)

print("ANCHOR:", anchor)
print("TARGET:", target)
print("COSINE SIMILARITY:", similarity)

ANCHOR: rich.markdown.Markdown.__init__
TARGET: rich.markdown.MarkdownElement.on_text
COSINE SIMILARITY: 0.9925876259803772


In [162]:
anchor = "rich.markdown.Markdown.__init__"
target = "rich.markdown.MarkdownElement.on_text"

node_ids, matrix = prepare_embedding_matrix(graph, embeddings)
node_to_index = {
    node_id: i for i, node_id in enumerate(node_ids)
}

# Nodes reachable from the anchor within 4 hops
nodes_by_id, outgoing, incoming = build_graph_index(graph)

reachable = graph_distance(
    [anchor],
    outgoing,
    incoming,
    max_depth=4
)

anchor_vec = matrix[node_to_index[anchor]]
anchor_norm = np.linalg.norm(anchor_vec)

scores = []

for node, distance in reachable.items():
    if node not in node_to_index:
        continue

    vec = matrix[node_to_index[node]]
    norm = np.linalg.norm(vec)

    if norm == 0 or anchor_norm == 0:
        continue

    similarity = float(
        np.dot(anchor_vec, vec) /
        (anchor_norm * norm)
    )

    scores.append((similarity, node, distance))

scores.sort(reverse=True)

target_rank = next(
    (i + 1 for i, item in enumerate(scores) if item[1] == target),
    None
)

print("REACHABLE NODES:", len(scores))
print("TARGET:", target)
print("TARGET RANK:", target_rank)

if target_rank:
    similarity, node, distance = scores[target_rank - 1]
    print("TARGET SIMILARITY:", similarity)
    print("TARGET DISTANCE:", distance)

    print("\nTOP 20 REACHABLE NODES:")
    for i, (sim, node, dist) in enumerate(scores[:20], 1):
        print(
            f"{i:2d} | sim={sim:.4f} | "
            f"dist={dist} | {node}"
        )

REACHABLE NODES: 675
TARGET: rich.markdown.MarkdownElement.on_text
TARGET RANK: None


In [164]:
import inspect

# Find likely experiment functions still present in the restarted runtime
names = [
    name for name in globals()
    if any(key in name.lower() for key in [
        "experiment", "evaluate", "run_all", "run_task", "benchmark"
    ])
]

print("Possible experiment functions:")
for name in names:
    obj = globals()[name]
    if callable(obj):
        print("-", name)

Possible experiment functions:


In [166]:
important = [
    "tasks_df",
    "load_resources",
    "build_graph_index",
    "graph_distance",
    "find_anchor_nodes_fast",
    "prepare_embedding_matrix",
    "embedding_retrieve_fast",
    "graphfix_expand",
    "extract_anchor_terms",
    "extract_changed_files",
]

for name in important:
    print(f"{name:30} {'✓' if name in globals() else '✗'}")

tasks_df                       ✓
load_resources                 ✓
build_graph_index              ✓
graph_distance                 ✓
find_anchor_nodes_fast         ✓
prepare_embedding_matrix       ✓
embedding_retrieve_fast        ✓
graphfix_expand                ✓
extract_anchor_terms           ✓
extract_changed_files          ✓


In [167]:
import inspect

GRAPHFIX_BASELINE_SOURCE = inspect.getsource(graphfix_expand)

graphfix_baseline = graphfix_expand

print("Baseline GraphFix frozen.")
print("Function:", graphfix_baseline.__name__)
print("Parameters:")
print(inspect.signature(graphfix_baseline))

Baseline GraphFix frozen.
Function: graphfix_expand
Parameters:
(graph, anchor_nodes, embedding_scores, budget=30, max_depth=3, alpha=0.55, beta=0.3, gamma=0.15)


In [169]:
def graphfix_v2(
    graph,
    anchor_nodes,
    embeddings,
    budget=30,
    max_depth=3,
    alpha=0.55,
    beta=0.30,
    gamma=0.15
):
    nodes_by_id, outgoing, incoming = build_graph_index(graph)

    # 1. Expand from the issue-derived anchors
    distances = graph_distance(
        anchor_nodes,
        outgoing,
        incoming,
        max_depth=max_depth
    )

    candidates = set(
        node for node in distances
        if node in nodes_by_id
    )

    # 2. Prepare embedding matrix
    node_ids, matrix = prepare_embedding_matrix(
        graph,
        embeddings
    )

    node_to_index = {
        node_id: i
        for i, node_id in enumerate(node_ids)
    }

    # 3. Calculate semantic similarity for EVERY graph-expanded candidate
    anchor_vectors = []

    for anchor in anchor_nodes:
        if anchor in node_to_index:
            anchor_vectors.append(
                matrix[node_to_index[anchor]]
            )

    if anchor_vectors:
        anchor_matrix = np.vstack(anchor_vectors)

        anchor_norms = np.linalg.norm(
            anchor_matrix,
            axis=1,
            keepdims=True
        )
        anchor_norms[anchor_norms == 0] = 1.0

        anchor_matrix = (
            anchor_matrix / anchor_norms
        )
    else:
        anchor_matrix = None

    scored = []

    for node in candidates:

        # Semantic relevance to the closest anchor
        semantic_score = 0.0

        if (
            anchor_matrix is not None
            and node in node_to_index
        ):
            vector = matrix[node_to_index[node]]

            norm = np.linalg.norm(vector)

            if norm > 0:
                vector = vector / norm

                similarities = (
                    anchor_matrix @ vector
                )

                semantic_score = float(
                    np.max(similarities)
                )

        # Structural relevance
        distance = distances.get(
            node,
            max_depth + 1
        )

        structural_score = (
            1 / (1 + distance)
            if distance <= max_depth
            else 0.0
        )

        # Keep the original constant diversity term
        diversity_score = 1.0

        total_score = (
            alpha * semantic_score
            + beta * structural_score
            + gamma * diversity_score
        )

        scored.append({
            "node": node,
            "score": total_score,
            "semantic_score": semantic_score,
            "structural_score": structural_score,
            "diversity_score": diversity_score,
            "distance": distance,
        })

    scored.sort(
        key=lambda x: (
            -x["score"],
            x["distance"],
            x["node"]
        )
    )

    # 4. Select compact context while discouraging repetition
    selected = []
    selected_modules = set()

    for item in scored:

        node = item["node"]

        parts = node.split(".")

        module = (
            ".".join(parts[:2])
            if len(parts) >= 2
            else node
        )

        repetition_penalty = (
            0.85
            if module in selected_modules
            else 1.0
        )

        item = dict(item)

        item["adjusted_score"] = (
            item["score"] * repetition_penalty
        )

        selected.append(item)

        selected_modules.add(module)

        if len(selected) >= budget:
            break

    return selected


print("GraphFix v2 created.")
print(inspect.signature(graphfix_v2))

GraphFix v2 created.
(graph, anchor_nodes, embeddings, budget=30, max_depth=3, alpha=0.55, beta=0.3, gamma=0.15)


In [171]:
task_4079 = tasks_df[
    tasks_df["instance_id"] == "rich_4079"
].iloc[0]

graph_4079, embeddings_4079 = load_resources(
    task_4079["repo"],
    task_4079["base_commit"]
)

terms_4079 = extract_anchor_terms(
    task_4079["problem_statement"]
)

anchor_candidates_4079 = find_anchor_nodes_fast(
    graph_4079,
    terms_4079
)

anchor_nodes_4079 = [
    item["node"]
    for item in anchor_candidates_4079[:5]
]

v2_result_4079 = graphfix_v2(
    graph_4079,
    anchor_nodes_4079,
    embeddings_4079,
    budget=30,
    max_depth=3
)

changed_files_4079 = set(
    task_4079["changed_files"]
)

hits_4079 = [
    item for item in v2_result_4079
    if node_file_path(item["node"]) in changed_files_4079
]

print("Task:", task_4079["instance_id"])
print("Terms:", terms_4079)
print("Anchors:", anchor_nodes_4079)
print("Selected nodes:", len(v2_result_4079))
print("Changed files:", changed_files_4079)
print("Hits:", len(hits_4079))
print("Coverage:", len(hits_4079) / len(changed_files_4079))

print("\nTop selected nodes:")
for item in v2_result_4079[:10]:
    print(
        item["node"],
        "| semantic:", round(item["semantic_score"], 4),
        "| distance:", item["distance"],
        "| score:", round(item["score"], 4)
    )

Task: rich_4079
Terms: ['github.com', 'Textualize']
Anchors: ['tests.test_ansi.test_decode_newlines', 'tests.test_cells.test_nerd_font', 'tests.test_console.test_capture_and_record', 'tests.test_console.test_recording_no_stdout_and_no_stderr_files', 'tests.test_console.test_render_lines_height_minus_vertical_pad_is_negative']
Selected nodes: 30
Changed files: {'rich/markdown.py'}
Hits: 0
Coverage: 0.0

Top selected nodes:
tests.test_console.test_recording_no_stdout_and_no_stderr_files | semantic: 1.0 | distance: 0 | score: 1.0
tests.test_ansi.test_decode_newlines | semantic: 1.0 | distance: 0 | score: 1.0
tests.test_cells.test_nerd_font | semantic: 1.0 | distance: 0 | score: 1.0
tests.test_console.test_capture_and_record | semantic: 1.0 | distance: 0 | score: 1.0
tests.test_console.test_render_lines_height_minus_vertical_pad_is_negative | semantic: 1.0 | distance: 0 | score: 1.0
rich.cells.cell_len | semantic: 0.9938 | distance: 1 | score: 0.8466
rich.console.Console | semantic: 0.9918

In [172]:
concept_terms_4079 = ["Inline", "table"]

concept_candidates_4079 = find_anchor_nodes_fast(
    graph_4079,
    concept_terms_4079
)

print("Concept terms:", concept_terms_4079)
print("Number of candidates:", len(concept_candidates_4079))

print("\nTop 15 candidates:")
for item in concept_candidates_4079[:15]:
    print(
        item["node"],
        "| score:", item["score"],
        "| matched:", item["matched_terms"]
    )

Concept terms: ['Inline', 'table']
Number of candidates: 20

Top 15 candidates:
rich.table.Table | score: 3 | matched: ['table']
rich.markdown.Markdown.__init__ | score: 2 | matched: ['Inline', 'table']
tests.test_markdown.test_inline_styles_in_table | score: 2 | matched: ['Inline', 'table']
benchmarks.benchmarks.TableSuite | score: 1 | matched: ['table']
benchmarks.benchmarks.TableSuite._print_table | score: 1 | matched: ['table']
benchmarks.benchmarks.TableSuite.time_table_heavy_wrapping | score: 1 | matched: ['table']
benchmarks.benchmarks.TableSuite.time_table_no_wrapping | score: 1 | matched: ['table']
examples.export.print_table | score: 1 | matched: ['table']
examples.fullscreen.Header.__rich__ | score: 1 | matched: ['table']
examples.fullscreen.make_sponsor_message | score: 1 | matched: ['table']
examples.print_calendar.print_calendar | score: 1 | matched: ['table']
examples.top_lite_simulator.create_process_table | score: 1 | matched: ['table']
rich.__main__.comparison | score

In [177]:
# Final reported results from the completed 127-task experiment

final_results = pd.DataFrame([
    {
        "Method": "Issue-derived anchors",
        "Mean Coverage": 0.417668,
        "Mean Precision": 0.107446,
        "Mean Context": 27.645669,
    },
    {
        "Method": "Embedding retrieval",
        "Mean Coverage": 0.407259,
        "Mean Precision": 0.087927,
        "Mean Context": 29.763780,
    },
    {
        "Method": "Fixed graph expansion",
        "Mean Coverage": 0.407106,
        "Mean Precision": 0.105407,
        "Mean Context": 29.023622,
    },
    {
        "Method": "GraphFix",
        "Mean Coverage": 0.414127,
        "Mean Precision": 0.093438,
        "Mean Context": 29.763780,
    },
])

display(final_results.round(4))

,Method,Mean Coverage,Mean Precision,Mean Context
0,Issue-derived anchors,0.4177,0.1074,27.6457
1,Embedding retrieval,0.4073,0.0879,29.7638
2,Fixed graph expansion,0.4071,0.1054,29.0236
3,GraphFix,0.4141,0.0934,29.7638


In [178]:
final_results_display = final_results.copy()

final_results_display = final_results_display.round({
    "Mean Coverage": 4,
    "Mean Precision": 4,
    "Mean Context": 2
})

display(
    final_results_display.style.hide(axis="index")
)

Method,Mean Coverage,Mean Precision,Mean Context
Issue-derived anchors,0.417700,0.107400,27.650000
Embedding retrieval,0.407300,0.087900,29.760000
Fixed graph expansion,0.407100,0.105400,29.020000
GraphFix,0.414100,0.093400,29.760000


In [180]:
paired_comparison = pd.DataFrame([
    {
        "Comparison": "GraphFix vs Issue-derived anchors",
        "Mean Δ Coverage": -0.003540,
        "Improved": 12,
        "Same": 98,
        "Worse": 17,
    },
    {
        "Comparison": "GraphFix vs Embedding retrieval",
        "Mean Δ Coverage": 0.006868,
        "Improved": 6,
        "Same": 112,
        "Worse": 9,
    },
    {
        "Comparison": "GraphFix vs Fixed graph expansion",
        "Mean Δ Coverage": 0.007021,
        "Improved": 8,
        "Same": 112,
        "Worse": 7,
    },
])

display(
    paired_comparison.round({
        "Mean Δ Coverage": 4
    }).style.hide(axis="index")
)

Comparison,Mean Δ Coverage,Improved,Same,Worse
GraphFix vs Issue-derived anchors,-0.003500,12,98,17
GraphFix vs Embedding retrieval,0.006900,6,112,9
GraphFix vs Fixed graph expansion,0.007000,8,112,7


# GraphFix: Adaptive Graph-Guided Context Selection for Local Software Engineering Agents

## Research Question

Given a small set of code symbols identified from a software-engineering issue, does adaptive graph expansion improve localization of the files modified by the reference patch while controlling context size?

## Hypothesis

Semantic retrieval can identify code that is lexically or semantically related to a software issue, but relevant software behavior is often distributed across callers, callees, dependencies, and related symbols.

We investigate whether combining semantic similarity with repository graph structure can provide a compact context that improves relevant-file localization compared with issue-derived symbol retrieval, embedding retrieval, and fixed graph expansion.

## Contributions

1. A graph-guided context selection strategy for local software-engineering agents.
2. An evaluation across 127 unique software-engineering tasks from the Gemma 4 Developer Agent benchmark.
3. A comparison of issue-derived anchors, embedding retrieval, fixed graph expansion, and GraphFix.
4. A diagnostic analysis showing how anchor quality affects downstream graph-based retrieval.

## Dataset and Resources

This study uses the resources released with the Google Gemma 4 Developer Agent benchmark.

The benchmark provides software-engineering tasks containing repository information, base commits, problem statements, reference patches, and test patches. For this study, we use the task problem statements and reference patches to evaluate code localization.

The benchmark also provides repository-level code graphs and precomputed code embeddings for the task base commits. The code graph represents relationships between code symbols, while the embeddings support semantic similarity between code nodes.

### Dataset statistics

- 129 benchmark task records
- 127 unique base commits
- 4 repositories:
  - FastAPI
  - Rich
  - Requests
  - HTTPX
- 127 matching graph resources
- 127 matching embedding resources
- 278 changed files across the evaluated tasks

Each task is evaluated against the files modified by its reference patch. These files provide the ground-truth targets for the localization experiment.

The graph and embedding resources are matched to tasks using the repository and exact base commit, ensuring that retrieval is evaluated against the corresponding repository state.

## Methodology

GraphFix treats repository code as a graph of interconnected symbols. The method consists of three stages: anchor discovery, graph expansion, and context selection.

### 1. Anchor discovery

The problem statement is converted into a small set of code-oriented terms. These terms are matched against graph nodes to identify candidate code symbols that may be relevant to the issue.

The highest-scoring candidates are used as initial anchors.

### 2. Semantic retrieval

For each anchor, precomputed code embeddings are used to measure semantic similarity with other repository symbols using cosine similarity.

This provides a semantic signal for identifying code that is related to the selected anchors.

### 3. Graph expansion

Starting from the anchor nodes, GraphFix traverses incoming and outgoing graph relationships up to a bounded depth.

This allows the method to include code that may not be directly retrieved by semantic similarity but is structurally connected to the initial anchors.

### 4. Adaptive context selection

Candidate nodes receive a combined score based on:

- semantic similarity to retrieved code,
- graph distance from the anchor set,
- a context-selection term.

The highest-scoring candidates are selected subject to a fixed context budget of 30 nodes. A repetition penalty is applied when multiple selected nodes belong to the same module.

The resulting set of nodes represents a compact repository context that can be supplied to a software-engineering agent.

### Baselines

We compare GraphFix with three retrieval strategies:

1. **Issue-derived anchors** — selects code nodes directly from issue-derived terms.
2. **Embedding retrieval** — retrieves semantically similar nodes from the precomputed code embeddings.
3. **Fixed graph expansion** — expands the issue-derived anchors through the code graph without adaptive ranking.

GraphFix uses the same repository graphs, embeddings, task base commits, and context budget as the comparison methods.

## Evaluation Metrics and Experimental Setup

The experiment evaluates whether each retrieval strategy identifies the files modified by the reference patch.

### Metrics

For each task, we measure:

**Coverage**

Coverage measures the fraction of reference-patch files represented by at least one selected code node.

\[
\text{Coverage} =
\frac{\text{Number of changed files hit}}
{\text{Total number of changed files}}
\]

A coverage of 1.0 means that all files modified by the reference patch were represented in the selected context.

**Precision**

Precision measures the fraction of selected nodes that correspond to files modified by the reference patch.

\[
\text{Precision} =
\frac{\text{Number of selected nodes from changed files}}
{\text{Total selected nodes}}
\]

**Context size**

Context size is the number of code nodes selected by each method. The target budget is 30 nodes.

### Experimental setup

The evaluation contains 127 unique task base commits. Each task is evaluated independently using the graph and embedding resources corresponding to its exact repository and base commit.

The same context budget is used across retrieval methods where applicable. Ground-truth localization is obtained from the files changed by the reference patch.

The reported results are macro-averaged across the 127 tasks, so each task contributes equally to the reported mean metrics.

## Results

The experiment compares four retrieval strategies across 127 unique task base commits.

The primary results are summarized using mean file coverage, mean precision, and mean context size. These metrics measure how much of the reference patch's changed-file set is represented, how focused the selected context is on changed files, and how much context is provided to the agent.

### Aggregate results

| Method | Mean Coverage | Mean Precision | Mean Context |
|---|---:|---:|---:|
| Issue-derived anchors | 0.4177 | 0.1074 | 27.65 |
| Embedding retrieval | 0.4073 | 0.0879 | 29.76 |
| Fixed graph expansion | 0.4071 | 0.1054 | 29.02 |
| **GraphFix** | **0.4141** | 0.0934 | 29.76 |

The issue-derived anchor baseline has the highest mean coverage (0.4177), while GraphFix achieves 0.4141. GraphFix therefore does not outperform the anchor baseline on aggregate coverage.

However, GraphFix achieves higher mean coverage than both embedding retrieval (0.4073) and fixed graph expansion (0.4071). Its mean coverage is approximately 0.0069 and 0.0070 higher than these two baselines, respectively.

The results indicate that graph-guided adaptive selection can provide a measurable benefit over purely semantic retrieval and fixed graph expansion, while also showing that the quality of the initial anchors remains an important factor.

### Paired task-level comparison

Aggregate means can hide task-level differences. We therefore also compare GraphFix with each baseline on the same individual tasks.

| Comparison | Mean Δ Coverage | Improved | Same | Worse |
|---|---:|---:|---:|---:|
| GraphFix vs Issue-derived anchors | −0.0035 | 12 | 98 | 17 |
| GraphFix vs Embedding retrieval | +0.0069 | 6 | 112 | 9 |
| GraphFix vs Fixed graph expansion | +0.0070 | 8 | 112 | 7 |

GraphFix improves coverage over embedding retrieval on 6 tasks and over fixed graph expansion on 8 tasks, while producing lower coverage on 9 and 7 tasks respectively. Against issue-derived anchors, GraphFix improves 12 tasks but performs worse on 17.

In all three comparisons, most tasks have identical coverage between the two methods. This suggests that the retrieval strategies frequently identify overlapping file-level context, while their differences become important on a smaller subset of tasks.

## Diagnostic Analysis

The aggregate results show that GraphFix provides modest gains over embedding retrieval and fixed graph expansion, but not over issue-derived anchors. To understand this behavior, we examined the `rich_4079` task.

The task's problem statement is:

> "Inline table code"

The reference patch modifies `rich/markdown.py`.

With the original anchor-extraction procedure, the extracted terms were `github.com` and `Textualize`. These terms produced generic test nodes as the highest-ranked anchors, causing subsequent graph expansion to begin from code that was not directly related to the changed file.

As a diagnostic, we instead tested the issue concepts `Inline` and `table`. These terms retrieved relevant repository symbols including `rich.table.Table`, `rich.markdown.Markdown.__init__`, and `tests.test_markdown.test_inline_styles_in_table`.

This analysis indicates that **anchor quality is a critical dependency of the graph-guided approach**. Graph expansion can only propagate useful structural information when its starting points are relevant to the underlying software behavior.

The diagnostic therefore identifies a concrete limitation of the current GraphFix pipeline: improvements to issue-to-symbol anchor extraction may be necessary before graph-guided context selection can consistently outperform simpler retrieval strategies.

## Limitations

This study has several limitations.

First, the evaluation measures **file-level localization**, rather than whether a software-engineering agent can successfully generate and validate a patch. High localization coverage therefore does not necessarily imply higher end-to-end task resolution.

Second, the experiment relies on the benchmark's **precomputed code embeddings**. These embeddings provide a useful semantic signal between repository symbols, but they were not generated directly from the natural-language problem statement.

Third, GraphFix depends on the quality of its initial anchors. The diagnostic analysis demonstrates that generic or poorly matched issue terms can lead graph expansion toward irrelevant regions of the repository.

Fourth, the current evaluation uses a fixed context budget of 30 nodes. Different context budgets or graph depths may produce different trade-offs between coverage and precision.

Finally, the benchmark contains a relatively small number of repositories, with many tasks concentrated in FastAPI and Rich. Results may therefore not generalize to all software projects or repository structures.

## Conclusion

This study investigated whether graph-guided context selection can improve localization of files modified by software-engineering reference patches.

Across 127 unique task base commits, GraphFix achieved a mean coverage of 0.4141, compared with 0.4177 for issue-derived anchors, 0.4073 for embedding retrieval, and 0.4071 for fixed graph expansion.

The results do not show that GraphFix universally outperforms the baselines. In particular, issue-derived anchors achieved slightly higher aggregate coverage. However, GraphFix achieved higher mean coverage than both embedding retrieval and fixed graph expansion, with mean improvements of 0.0069 and 0.0070 respectively.

The paired analysis also shows that most tasks have identical coverage across methods, while a smaller subset benefits from graph-guided selection. The diagnostic analysis further indicates that the quality of the initial code anchors is a key factor determining whether graph expansion reaches relevant repository regions.

Overall, the findings support graph structure as a useful complementary signal for local software-engineering context selection, while highlighting anchor discovery as an important area for future work. Future versions could improve natural-language-to-code-symbol grounding, learn adaptive graph depth and context budgets, and evaluate the resulting contexts directly within end-to-end software-engineering agents.

## References

1. Google. *Gemma 4 Developer Agent Benchmark*. Kaggle, 2026.  
   https://www.kaggle.com/competitions/gemma-4-developer-agent

2. Google. *Gemma 4 Developer Agent — Paper Track*. Kaggle, 2026.  
   https://www.kaggle.com/competitions/gemma-4-developer-agent-paper

3. Yang, J., Jimenez, C. E., Wettig, A., et al. *SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering*. Advances in Neural Information Processing Systems (NeurIPS), 2024.

4. Zhang, F., Chen, G., et al. *RepoCoder: Repository-Level Code Completion Through Iterative Retrieval and Generation*. Proceedings of the 2023 Conference on Empirical Methods in Natural Language Processing (EMNLP), 2023.

5. Guo, D., Ren, S., Lu, S., et al. *GraphCodeBERT: Pre-training Code Representations with Data Flow*. International Conference on Learning Representations (ICLR), 2021.

6. Feng, Z., Guo, D., Tang, D., et al. *CodeBERT: A Pre-Trained Model for Programming and Natural Languages*. Proceedings of the 2020 Conference on Empirical Methods in Natural Language Processing: Findings, 2020.

In [184]:
import pandas as pd

final_results = pd.DataFrame([
    {
        "Method": "Issue-derived anchors",
        "Mean Coverage": 0.4177,
        "Mean Precision": 0.1074,
        "Mean Context": 27.65
    },
    {
        "Method": "Embedding retrieval",
        "Mean Coverage": 0.4073,
        "Mean Precision": 0.0879,
        "Mean Context": 29.76
    },
    {
        "Method": "Fixed graph expansion",
        "Mean Coverage": 0.4071,
        "Mean Precision": 0.1054,
        "Mean Context": 29.02
    },
    {
        "Method": "GraphFix",
        "Mean Coverage": 0.4141,
        "Mean Precision": 0.0934,
        "Mean Context": 29.76
    }
])

display(final_results)

,Method,Mean Coverage,Mean Precision,Mean Context
0,Issue-derived anchors,0.4177,0.1074,27.65
1,Embedding retrieval,0.4073,0.0879,29.76
2,Fixed graph expansion,0.4071,0.1054,29.02
3,GraphFix,0.4141,0.0934,29.76


### Aggregate results

| Method | Mean Coverage | Mean Precision | Mean Context |
|---|---:|---:|---:|
| Issue-derived anchors | 0.4177 | 0.1074 | 27.65 |
| Embedding retrieval | 0.4073 | 0.0879 | 29.76 |
| Fixed graph expansion | 0.4071 | 0.1054 | 29.02 |
| **GraphFix** | **0.4141** | 0.0934 | 29.76 |

The issue-derived anchor baseline has the highest mean coverage (0.4177), while GraphFix achieves 0.4141. GraphFix therefore does not outperform the anchor baseline on aggregate coverage.

However, GraphFix achieves higher mean coverage than both embedding retrieval (0.4073) and fixed graph expansion (0.4071). Its mean coverage is approximately 0.0069 and 0.0070 higher than these two baselines, respectively.

The results indicate that graph-guided adaptive selection can provide a measurable benefit over purely semantic retrieval and fixed graph expansion, while also showing that the quality of the initial anchors remains an important factor.